# DIMER Capstone: Philippine Reef Heat-Stress Outlook

**Profile:** TASK-INFERENCE · **Mode:** GUIDED · **Standard:** NOTEBOOK_SPEC 2.2
**Status:** Candidate pending fresh Colab T4 qualification.

Can a pretrained model improve short-horizon forecasts of Philippine regional heat stress over simple baselines?
You will forecast daily HotSpot at 7, 14 and 28 days, calculate accumulated Degree Heating Weeks (DHW),
compare errors, and export evidence. A baseline winning is a valid scientific result.

**Prerequisites:** basic Python and Colab familiarity; no prior ML experience.
Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. Any current Colab Python version works:
setup downloads a pinned, managed Python 3.12.13 into an isolated environment, so you do not need to choose an
older runtime version. No tokens, uploads, repository clone, manual restart or configuration edits are required.
Setup and model downloads need internet access.
The target budget is 30 minutes and 12 GiB GPU memory; these limits are unverified until hosted qualification.
The five NOAA text files are embedded as a verified 1 MB data archive; approximately 0.48 GB of model weights
and a larger Python environment are downloaded. Leave several GB of runtime disk free.

**Input → System → Output:** daily regional HotSpot history → baseline/Chronos forecasting → DHW accumulation
and chronological evaluation → regional outlook, comparison tables and reproducibility bundle.

This is an educational retrospective stress-indicator benchmark, not NOAA's official outlook, a reef-level
measurement, an observed bleaching prediction or an operational warning. Forecasts use a frozen September 2026 snapshot.

**AI Assistance Disclosure:** Generative AI assisted code development and technical writing under maintainer
direction. The maintainer remains responsible for implementation review, validation and release decisions.
AI assistance is not independent verification, provider endorsement or release approval.

## Roadmap and learning objectives

1. Explain what a satellite-derived regional HotSpot represents.
2. Reconstruct accumulated stress and distinguish known heat from predicted heat.
3. Compare models on identical chronological origins without future-data leakage.
4. Change one setting on validation data and interpret uncertainty and failure cases.
5. Export evidence, verify it in a fresh process, and write a bounded conclusion.

**Core concepts** explain the science; **Evaluation practice** explains comparisons; **Infrastructure** cells
handle environment, data integrity and exports. Infrastructure code is visible but collapsed where supported.
Run it without studying every line. Checkpoint answers and completion notes never block execution.

## 1. Core concepts: what do the numbers mean?

**SST** is sea-surface temperature. A **HotSpot** is positive temperature excess over a local warm-season reference,
in °C. **DHW** sums HotSpot values at or above 1 °C during the last 84 days and divides by seven, giving °C-weeks.
The value 1.00 contributes; 0.99 does not. An **origin** is the last observed date; **horizon** is the number of future days.

NOAA's regional series uses the pixel at the region's 90th-percentile HotSpot each day. The representative pixel
can change. These are not regional mean temperatures or stationary reef sensors. Do not subtract the header's
regional average climatology from a selected pixel's SST to invent a new HotSpot.

**Predict:** could DHW remain high when today's HotSpot is zero?
<details><summary>Show a sample answer</summary>Yes. DHW retains qualifying heat from the previous 83 days.
Current conditions alone do not describe accumulated exposure.</details>

Source: [NOAA regional methodology](https://coralreefwatch.noaa.gov/product/vs/map.php) and
[DHW method](https://coralreefwatch.noaa.gov/product/5km/methodology.php).

In [ ]:
# Infrastructure: isolated pinned environment on a managed Python 3.12; the kernel is unchanged.
import sys, subprocess, pathlib, json, hashlib, base64, os, time, platform
SETUP_STARTED = time.perf_counter()
TARGET_PYTHON = "3.12.13"  # uv-managed interpreter, independent of the Colab kernel's Python
RUN_ROOT = pathlib.Path.cwd() / "reef_capstone_run"
RUN_ROOT.mkdir(exist_ok=True)
ENV_ROOT = RUN_ROOT / "environment"
PYTHON = ENV_ROOT / "bin/python"
LOG_DIR = RUN_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
def child_environment():
    # Children never inherit the kernel's Python path, startup file, inline plotting backend or
    # Colab's UV_SYSTEM_PYTHON (which makes uv warn that --system has no effect on uv venv).
    dropped = {"PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "UV_SYSTEM_PYTHON"}
    env = {k: v for k, v in os.environ.items() if k not in dropped}
    env.update(MPLBACKEND="Agg", PYTHONUNBUFFERED="1")
    return env
def run_logged(command, log_name, echo=True):
    # Run a child with its output in logs/<name>.log. Poll it, echo new lines, and print a
    # heartbeat every 30 s so a long silent step is visibly alive; on failure show the log tail.
    started = last_beat = time.perf_counter()
    log_path, shown, returncode = LOG_DIR / f"{log_name}.log", 0, None
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen([str(part) for part in command], stdout=log,
                                   stderr=subprocess.STDOUT, env=child_environment())
        while returncode is None:
            try:
                returncode = process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                pass
            text = log_path.read_text(encoding="utf-8", errors="replace")
            lines = text.splitlines()
            complete = lines if text.endswith("\n") or returncode is not None else lines[:-1]
            if echo:
                for line in complete[shown:]:
                    print(line, flush=True)
            shown = max(shown, len(complete))
            if returncode is None and time.perf_counter() - last_beat >= 30:
                print(f"  ... {log_name} still working, {time.perf_counter() - started:.0f} s", flush=True)
                last_beat = time.perf_counter()
    if returncode:
        tail = "\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-30:])
        raise RuntimeError(f"{log_name} failed with exit code {returncode}; log: {log_path}\n{tail}")
    return time.perf_counter() - started
run_logged([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.10.12"],
           "setup_uv", echo=False)
def environment_python():
    if not PYTHON.exists():
        return None
    probe = [str(PYTHON), "-c", "import platform; print(platform.python_version())"]
    return subprocess.run(probe, capture_output=True, text=True).stdout.strip() or None
ENV_REUSED = environment_python() == TARGET_PYTHON
if not ENV_REUSED:
    run_logged([sys.executable, "-m", "uv", "venv", "--clear", "--managed-python",
                "--python", TARGET_PYTHON, ENV_ROOT], "setup_python")
if environment_python() != TARGET_PYTHON:
    raise RuntimeError(f"Isolated environment is not Python {TARGET_PYTHON}; preserve this log.")
REQUIREMENTS = '# This file was autogenerated by uv via the following command:\n#    uv pip compile tools/reef-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes -o tools/reef-requirements.lock\naccelerate==1.15.0 \\\n    --hash=sha256:5654f8c5eaa0d4fa68b33e287a97765da6849bf6d51dcac874e73fbbddfb6134 \\\n    --hash=sha256:97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec\n    # via chronos-forecasting\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via requests\ncharset-normalizer==3.5.1 \\\n    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \\\n    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \\\n    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \\\n    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \\\n    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \\\n    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \\\n    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \\\n    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \\\n    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \\\n    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \\\n    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \\\n    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \\\n    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \\\n    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \\\n    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \\\n    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \\\n    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \\\n    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \\\n    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \\\n    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \\\n    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \\\n    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \\\n    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \\\n    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \\\n    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \\\n    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \\\n    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \\\n    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \\\n    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \\\n    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \\\n    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \\\n    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \\\n    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \\\n    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \\\n    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \\\n    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \\\n    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \\\n    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \\\n    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \\\n    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \\\n    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \\\n    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \\\n    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \\\n    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \\\n    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \\\n    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \\\n    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \\\n    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \\\n    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \\\n    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \\\n    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \\\n    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \\\n    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \\\n    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \\\n    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \\\n    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \\\n    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \\\n    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \\\n    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \\\n    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \\\n    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \\\n    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \\\n    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \\\n    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \\\n    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \\\n    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \\\n    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \\\n    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \\\n    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \\\n    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \\\n    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \\\n    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \\\n    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \\\n    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \\\n    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \\\n    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \\\n    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \\\n    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \\\n    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \\\n    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \\\n    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \\\n    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \\\n    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \\\n    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \\\n    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \\\n    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \\\n    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \\\n    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \\\n    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \\\n    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \\\n    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \\\n    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \\\n    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \\\n    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \\\n    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \\\n    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \\\n    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \\\n    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \\\n    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \\\n    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \\\n    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \\\n    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \\\n    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \\\n    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \\\n    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \\\n    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \\\n    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \\\n    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \\\n    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \\\n    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \\\n    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \\\n    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \\\n    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \\\n    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \\\n    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \\\n    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \\\n    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \\\n    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \\\n    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \\\n    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \\\n    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \\\n    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \\\n    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \\\n    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \\\n    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \\\n    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \\\n    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \\\n    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \\\n    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \\\n    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \\\n    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \\\n    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \\\n    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \\\n    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \\\n    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \\\n    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \\\n    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \\\n    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \\\n    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \\\n    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \\\n    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \\\n    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \\\n    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \\\n    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \\\n    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \\\n    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \\\n    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \\\n    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \\\n    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \\\n    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \\\n    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \\\n    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \\\n    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \\\n    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \\\n    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \\\n    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \\\n    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \\\n    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \\\n    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \\\n    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \\\n    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \\\n    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \\\n    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \\\n    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \\\n    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \\\n    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \\\n    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \\\n    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \\\n    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \\\n    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \\\n    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \\\n    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f\n    # via requests\nchronos-forecasting==2.3.1 \\\n    --hash=sha256:785116f3f10aaac44a315700c5e4de71a5fe6385c9c771722d688f402fe75083 \\\n    --hash=sha256:d9d00ec9b1621235bfb26685638bf054885f4c000863678f1c775dfab2697496\n    # via -r tools/reef-requirements.in\ncontourpy==1.4.0 \\\n    --hash=sha256:018227e134b73090b06f911e773a526c77c9af426be29065500ce096a5accd4c \\\n    --hash=sha256:072a702e2e178f4fcf96f04775d0c059e4c917925abf0d3208ebb6865df3c23d \\\n    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \\\n    --hash=sha256:107ec46f7aa1664266d69b1181aadc953d58d39f7d8d648fa5b795d4a060b0de \\\n    --hash=sha256:181bea01bc742734ae672fa00c717d855dd35e1f029536406538c52e7b0cd73d \\\n    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \\\n    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \\\n    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \\\n    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \\\n    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \\\n    --hash=sha256:2e741a39dfc96babe1722561e81351aeec104526c57dbe91c167c1db606c2d55 \\\n    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \\\n    --hash=sha256:3e0a2392533e5e2abdf277c951047516acc8d2f3c42b2dd9dc34c907c9ebabf2 \\\n    --hash=sha256:404dbcd9233513dfd1323b66ea593fef90e899f658f6b8a9ed8e93bd0ca669db \\\n    --hash=sha256:417be048f7e122cefbe2a34c51a1f2b0410f8eb35796f794d45475ddb7872d9d \\\n    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \\\n    --hash=sha256:432e89f835cf01f8d2123a130a3126b17ee6e0348e4d62c0973872b30aa873bf \\\n    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \\\n    --hash=sha256:43d80072a32299bf945de0a6dd4e034ea0162e832261de2e07a274a5adbba9c9 \\\n    --hash=sha256:4bffcb2e8cc5c0e837631acab59dd04265fdac376550e7c3e310523877c68cb7 \\\n    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \\\n    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \\\n    --hash=sha256:5545ff38229c15d11d822fc6f7415b932da9f4975f53e305b7b96ebfb01f889c \\\n    --hash=sha256:5a4c89c7f38a0d7a94356d74e3391073c4325c4d47ad2fd065c3fe7dbae16c0c \\\n    --hash=sha256:61624f6722480aa7e168fd746164e7bfdf48f8ccb6542f1613741200dc37e2b1 \\\n    --hash=sha256:618137ec5778fb76d494c9ec854b104a377fddba0d128bef7136cce2c3bf0df8 \\\n    --hash=sha256:630d32f06cedf37b7f1dd2b6a12e4492826bd477a4c9c7b2e2fee6ce3a2cea76 \\\n    --hash=sha256:64039341e2d8804f1a13bda8c69083eab935167bbd7e856cf096241f17d5bd45 \\\n    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \\\n    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \\\n    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \\\n    --hash=sha256:6f963ea9f0d68d2b6a02d861e3c0aae09ae25ba21f243fcf281e9fed468b078d \\\n    --hash=sha256:738c44fa71735a617f36da58e32810512407d283d5d1bb5b1cecb00d7eeba7cf \\\n    --hash=sha256:758e7496cec3195fc28a28945eab3ceef348c47fb95c7afc2eca2f3901209c8c \\\n    --hash=sha256:78ed5c5f962b3e156109f0531b174a65ba80d13cc681a70227b12524c162e184 \\\n    --hash=sha256:79b06c60d5e569ce12c5da8f0c51217cdc484386e2349fa717c30fafb56d2871 \\\n    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \\\n    --hash=sha256:80c7fc8e7ac217777ce7ba2db3cf8478652b5a4b335283111012ff9eaac9d842 \\\n    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \\\n    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \\\n    --hash=sha256:90feb8da006803a95ba573fddc9536357d0c3faddc3a4c9e22816ca49af52878 \\\n    --hash=sha256:912c6afaa106e2f74ba22b30416b77ef7eb94e3bdc5a4df51ab147845dba9b6d \\\n    --hash=sha256:96019f059bcb3774acc0104be3360a57b36d33eaaa2e1d7f77c800ca8e07618b \\\n    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \\\n    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \\\n    --hash=sha256:a1c8a74744fa746eeaa12f0f9ed7f8b4add9d8f1b14d95e3b5e133675eac888f \\\n    --hash=sha256:a3e67bc1a6a4618a1dac7c3053a9ffece5ddfa2046b2670b342cf430bb0b87d1 \\\n    --hash=sha256:a58b5f130afec61a093d743cdda435cc4047748b6791627e34a68f41ec3a9b07 \\\n    --hash=sha256:a5e2bb871b90cd7a52bdee63bf80ef7acb46398d27efc66d7679240016efc5aa \\\n    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \\\n    --hash=sha256:b7794ea07cab575633daad8d6e963b662053391022294d3aacc1d9ba9ef54114 \\\n    --hash=sha256:b9dc493e7924e5aa32d89d0b8e50986280f3d4b284968a792787bd7bcffd3cad \\\n    --hash=sha256:be85d160e7c795113c2a93254dde2fc7e86b375322570eee78b420969426973c \\\n    --hash=sha256:bee6ff96653e0807cd0f8eaef00174a031e3a2effa5c20f49a5a8d574b05af23 \\\n    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \\\n    --hash=sha256:c8be3392b84cf43373a488874dc1ebf46f60784a70977f92586e18019d71c7ed \\\n    --hash=sha256:c927ec747633e68c9920bcaad48b0bcefe648812b822f18c34e2d0e2270a3d2a \\\n    --hash=sha256:cc87f2ffa84a49a77d76cb792f28a48ff3b05a222aa0b2bddbd839ba78e1d5ca \\\n    --hash=sha256:dd59df9e2fb0aff8bd7fd1adb9f349b7c835245a9d0f18c2f2deeb537190f2b1 \\\n    --hash=sha256:ddf5a2596d716fd3793434844caf31abc7a40b1e8718431420c89858268fb909 \\\n    --hash=sha256:de503609fdb71f597634ca64fb4bd2f13d27c97f140e912b1e8ed93544840df0 \\\n    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \\\n    --hash=sha256:e439ab450c93455feb0218532ded3e946ec7a9b5f459069f122cfa44b89229f6 \\\n    --hash=sha256:ea09dc704029930cbd097f75cb0cc1db9852adbcf1a151fac7e9559f7bcbfd19 \\\n    --hash=sha256:ef47fef9a912c77e3d84b014f701e5a9340f25703e9ed3bcebe37f91fb69dc49 \\\n    --hash=sha256:ef9440f6f8506246269a82734f5ff9e2e4c5e775b3996fc883cc491c5257eca6 \\\n    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \\\n    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3 \\\n    --hash=sha256:fa1b787362a3856e63dd2b89449f6c384d55ee1beb8f94f4bcc871b40f02462c \\\n    --hash=sha256:fc9feef8f1f001c5b87decadc67c4a5d1eebb62ca39c4763d1237ff62cf2b707\n    # via matplotlib\ncycler==0.12.1 \\\n    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \\\n    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c\n    # via matplotlib\neinops==0.8.2 \\\n    --hash=sha256:54058201ac7087911181bfec4af6091bb59380360f069276601256a76af08193 \\\n    --hash=sha256:609da665570e5e265e27283aab09e7f279ade90c4f01bcfca111f3d3e13f2827\n    # via chronos-forecasting\nfilelock==4.0.5 \\\n    --hash=sha256:2b155f098c4f285fb41954a22c616c4e8a0635b78c184338ba3023c1c91a4b4d \\\n    --hash=sha256:be9166068623636cedcad7b2ef735b04e5fb4344d978e09fd6fbd6ede06f14f0\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfonttools==4.66.0 \\\n    --hash=sha256:02117d05dddb51e39b5c5a6eed0702ed8e0e1c341cb138101b2446eb479858f8 \\\n    --hash=sha256:03922992966a1830b94a750961d1ccfe1a7375d792ca61077a5afb719811788a \\\n    --hash=sha256:056641fcedeaad24b92e343e35e9fd1337200c102450678894b17928482d010d \\\n    --hash=sha256:0640e69b00e089d6b82e0a54e3a03af84b775f1f0b2cbecc1b0d7fe3980a194a \\\n    --hash=sha256:0cf33d1041364ebb7a9958db757e213c6d2a238eb41755fbf935db028c2db20f \\\n    --hash=sha256:107aae7c38f05561fef5103baa98a44400b5f4c69d87839c9a5f8125405e7d9a \\\n    --hash=sha256:135ce8738a6341fbefeedc061e20c2889350e84405abea7e61b1c9d87e1a1f2f \\\n    --hash=sha256:15315302c620e6ce8582934e07932dcf442fc3f59ab208d30564200b9a2f7ea4 \\\n    --hash=sha256:1d120ea0f5260b04e9b5ac0d9239efde4c23d990347d5d5889cdd37221726fec \\\n    --hash=sha256:221defad3b1949c1fdde2558d1fe780d42f926b683a71a46a4495a0e6a6614bd \\\n    --hash=sha256:2b278e2abe596872b3c9fe611f956ded85588537974e48e2adbe2d6ee159e099 \\\n    --hash=sha256:35684b562df7154d7a0ebfa512db9199c7fb0a93b5abfb59dac9d022dbee7aaa \\\n    --hash=sha256:379b94fe10651d26caf398eb646b3ecde886a9c504d3cce5e46164d58a78e32d \\\n    --hash=sha256:3889fdbe63e85dbf2a0d9d7c9d90746827ce60af2cdd24f0bb4738380a9540eb \\\n    --hash=sha256:398bfc5ce9055e0b91a6189f8cf462d4230b38f522b2ceaecb4f34cbf4fb5480 \\\n    --hash=sha256:3ea59aaec135c96c4cff20d79fdf1d10623993712a77990585a42cfd6a42bc5c \\\n    --hash=sha256:4513753259f06316bdc2039c9bf3129e74923c1cfad0722b4728a9ce19beeeb2 \\\n    --hash=sha256:4bdaf80590fcccb1b9d8e3b8ca72225defb14d03b79788f59781ce1bb66049b9 \\\n    --hash=sha256:5dda4087a7f8f0b985995489b3c0115bd117f18c0fc236013a0e78e837bb9ac2 \\\n    --hash=sha256:6371e7ed43cc9e23550fe4a1b72af3205275f28dcecac8c09b80a4582e326802 \\\n    --hash=sha256:6627f48130cdbe04c7e1189bff4192886ecda757f575626ee8468f6580eaecf3 \\\n    --hash=sha256:67ea5af3ca60e1e5c9b2841b1f6dba5ef16aab581bf463abb68515447154ab61 \\\n    --hash=sha256:6b11180c4166a4fc353bb02b147322b7c454927bba4f5040ccdef109a73f9051 \\\n    --hash=sha256:700c448f190cc7bbd9f059bd96131f4770e90938c9692550b142153bea73b14c \\\n    --hash=sha256:8009e736e1491c569392b2e9782c30dcb6ac879def102c8c2e06add8d2adc10f \\\n    --hash=sha256:850d50b347f31667d3a277a6e66a4219236db9d98afadb2993b8addf9bd4f593 \\\n    --hash=sha256:85e2960a29e6882567b981ba408b5a5ca2d8865b8f67240e1b420348997a13be \\\n    --hash=sha256:8739d77810aeb6d2ecd55cc2e4864d98a00c317fb6131a6d22ca58b3f50cf22a \\\n    --hash=sha256:886f3029ea10362c592d2e1a5326a44645b85a2a0be6fdb7c9edbdb60cedb0ab \\\n    --hash=sha256:8af85dfc2ad564f95b3e650024578bb2268bc4d827d418aab9650bbde4b6926a \\\n    --hash=sha256:8ea6265535b25f8254868f5a212a1c9dbd88c2c4050f3df18d9174949df0d6d9 \\\n    --hash=sha256:9b0198962bbba64a88bf15dc5d95e4bab8c5208dede22a10fb06280406ed9874 \\\n    --hash=sha256:a5a0624aaefb0a29e4ed7500a98732a1b0ec2d3040fe86cc8837f82e3fe4f446 \\\n    --hash=sha256:a9a45a23b020e667499fb2dbce0f1373b1b1b8cf88e47f3031e69ef3ea21f3ee \\\n    --hash=sha256:ada87643b20a8fa5763e6dd4fa0f5e2901cbdbe0833deb6d5c5a9c3186782b94 \\\n    --hash=sha256:aea0cc5609a5f2a2500f91bd6e1989fdd22da0f225be00dc1c4cec775838a7d0 \\\n    --hash=sha256:b02686b2e47115b2378ab6156abd984c776c6faa4f56898fbcc1c30ee387937d \\\n    --hash=sha256:b432b6b54a0f3be118c6243312c826aa086aa9bcb150160be4f0bc8a21ea3aaa \\\n    --hash=sha256:b46500e4d6f5708a9127ea12902ba54cb0ec594fc6bec649afc53154a1125b2b \\\n    --hash=sha256:ba65eb3e2c46ad0922d84ef78e287c2f48b7a8cae4420728bf3b8e583282d6d7 \\\n    --hash=sha256:bc7b7ddc1a1f46c363354304e9a8dd93722e4a6a24f785015650898dacf40df9 \\\n    --hash=sha256:c144e68572af1dfcb05fe065827d0920d4118b265504928e1d2262e3701a25a1 \\\n    --hash=sha256:c3a66c749b69e9abd92e519ccbdea7210522b0fffdba3f6492a8fab461d4708a \\\n    --hash=sha256:c519433e8632284dbe64ee4c81b609a17e205bc1afff285553723964880d1c89 \\\n    --hash=sha256:ce818581070527883e3678b92b1ac7ed3569e48c80f51b89b264f97d84be2608 \\\n    --hash=sha256:cf0033f343cb1592fb24ae3eaf8b58156f208884ac16991094293ee94c585654 \\\n    --hash=sha256:cffa168522a0e057d3a5628c8ebd43ddf66a34613f42e116bffb85b0b32ce5a0 \\\n    --hash=sha256:d2d10f89c4bf7cd42b84695da617f9130fa594647ee74673bf8c8d001fa04ef9 \\\n    --hash=sha256:dbee639a23c4e067aedfbcf84d2e466a71a1a1156ad04d6d09023791ed167495 \\\n    --hash=sha256:e4677025ee40b3b1420387ca85f55545b8095f7596c3d5d59552dbee19c91f07 \\\n    --hash=sha256:ee4f85d1341af630d787514eef1a0e1ad883f9d6c11a4c1465faeb9cd0bb5ced \\\n    --hash=sha256:ee7b9f6c835ea1a9beb5a35c8eb0c62b6a3290f8105d3d0c29a0a18c90f9b8ba \\\n    --hash=sha256:ef0610dfe7bb5bf574d9bdad6f597403ebc9807d124ac6f148d7604b2609be98 \\\n    --hash=sha256:f014bbf05f30731bf8b9f5c7d2b4c0e4f28f375926f95b187f85e0bbceb16029 \\\n    --hash=sha256:f2032ac005e14c56e774fd19f2bc1c22c796c0c3b39d91741c0e3f9575944d38 \\\n    --hash=sha256:fa995d96ced196388d8ed7a67ee001b181459d230e24bb68b193ae859f5609da \\\n    --hash=sha256:fd3dcb69d65c05a2fffa5d9b743b77e0cdc13832665ba435065493dad2b57f83 \\\n    --hash=sha256:fd9297a5f670f3e59ae697289ddad5902d3c4a86188e83e7823bde06334a560c\n    # via matplotlib\nfsspec==2026.9.0 \\\n    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n    # via\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhuggingface-hub==0.36.2 \\\n    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n    # via\n    #   -r tools/reef-requirements.in\n    #   accelerate\n    #   tokenizers\n    #   transformers\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via requests\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\nkiwisolver==1.5.1 \\\n    --hash=sha256:007a5553dfc4f4e8d184f588a0200e2cd4b63a59cc8796df3c39909e679dc7a0 \\\n    --hash=sha256:0324cd2567259b7a095f6cf18a52b0ffc6f3de9e69528ff1bc0e7a37bd43ff1a \\\n    --hash=sha256:0627b9bceb9c3cdcf12b8a18655eedfed2692b038df27423383c120d0b7dc2d6 \\\n    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \\\n    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \\\n    --hash=sha256:0a4faea5c6db201c6a21391d2ac926ea97acf7dacdbc3c417189e1adb1a00837 \\\n    --hash=sha256:0ba9527afc80ae3d7814ed98b6572d02bf85eaf48065678342c5f0c6dab7a8c7 \\\n    --hash=sha256:0d8924877ce22e17326a99a418c3c82037da078df3c6a260b13eca677444e6e7 \\\n    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \\\n    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \\\n    --hash=sha256:16895f553ee6620a827d2da56b871f835fb70b9216cca5d188e885caf6e3bd23 \\\n    --hash=sha256:17851e5dad4484be0cbccbde3b15331deae036de9aebd45eed964487802b172f \\\n    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \\\n    --hash=sha256:18170a77ddfecf40ec60d0928268dc95880c881864e015a8f34094ed18b9b9ad \\\n    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \\\n    --hash=sha256:18a0cfb124546a4c2e6087c5f3029c7f44b37c85b142e0ced71f73a7599ac208 \\\n    --hash=sha256:1983f0974a750a6f6556f368ba11105d1d8369c735b944747c9f12ae5aea7aae \\\n    --hash=sha256:1a7587dc335f2c0f5bd577fd0540bd16c66006bdb60f759a1059f025e6c4f071 \\\n    --hash=sha256:1acc7e5b7ef05e9da8bb70cd6c7c4513090213d2e1ad9720f599f0bf6c52aec5 \\\n    --hash=sha256:1d852545c4d0e35a72728d072cbaa59e2fa7dd84bdf01e068d670dd0ceb58eb6 \\\n    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \\\n    --hash=sha256:1fff05e239575b1481b6ed1a782f6fad616efbf1f0b1f44e6e85c4dfe426e483 \\\n    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \\\n    --hash=sha256:22d5e5aaad6be121f2515765e3b1c444352cb8eb4c86510801db8f2e50757316 \\\n    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \\\n    --hash=sha256:255605693a483db7bd5c79f60437f7bf658f7f520d61aa42722e32257c941951 \\\n    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \\\n    --hash=sha256:27add358abe374ebaa3b8763ef380bc99051b5a4b18d94878366a9e4f59efef0 \\\n    --hash=sha256:2ae70bc59790d2af72a3f76f24b272403e135070340281108b447cb77ea70819 \\\n    --hash=sha256:2e10ae1bba1899188b33557c10d73affcc12033edd18adddb57d209039976a4c \\\n    --hash=sha256:3221f78211074f561c44ca42eac0619828171bec15a2c4cf6f7747d07df76e8e \\\n    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \\\n    --hash=sha256:34ec467940442c9943016fb2d4c81d1ba84351eeca2f1a78f8bc87f1ba0d414c \\\n    --hash=sha256:37f801b5d7cc0e5a548921308e059fd2b057bb42972b591cfa3049f95423c4ed \\\n    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \\\n    --hash=sha256:3c24cd69455e1b00ddf770c13b6e2c33e07d6dc3f2d34add0bf9277c5c6bbd46 \\\n    --hash=sha256:3cc210010fd2f438a3ed430b45f1b501fd13a8618bf984dc2c5ce5b69b78752e \\\n    --hash=sha256:3fa5855898f6d3d01b72ccd48a2d65cbdee301251603fefe34e2025bddba219c \\\n    --hash=sha256:416ba7ff9f233b7036689bb5a3783537e838ad483f63558d2a800f75afe738b1 \\\n    --hash=sha256:431dc224a1a92a5c8f582d96e505196a3b5997a7271076678da2dfde67b77e9a \\\n    --hash=sha256:43844c1a7ad6d723d5b5b4c4fc7f5bd399c40e288120d16257c7c9e8765c6e85 \\\n    --hash=sha256:44b8faef94f1857e77fa0238f3390ff1ac51d2ea20a487e2e452a59fd2b5f5ca \\\n    --hash=sha256:470d420f98d368d6f010633a20659b544c5fdfa5329e6b70219f2ef08fd4a7ef \\\n    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \\\n    --hash=sha256:4d4ca09bf13cff792b1884f64b98ee6c2467930d632233be25c56b442d99f10e \\\n    --hash=sha256:5025e36fb4fb275cef0a4e30dbb11cb4ae61d1c83deb90189cb5d7e4cafd6b55 \\\n    --hash=sha256:509735237ae0d849e8a843551d423d2500d2e0a9ac1611a145658b29c0fb9f85 \\\n    --hash=sha256:534f02c1abb31ed6dbd3515545285c330b2f12d00fdb1fdb71658b9ca5a13a6a \\\n    --hash=sha256:5978c3340f16a35c30f8ab2fa7bcf559973c55f1a5ef6970e1f621acf3c4db13 \\\n    --hash=sha256:5b973887ff782cfd6b67c9904ad8ca542e0bc5e4961503408b423b5a688b4d38 \\\n    --hash=sha256:5c490db2168a508088f59140dd392556a54b8bd1048fc6383c8baff13c359673 \\\n    --hash=sha256:5d142e352eb13facc7dd047489aebdff6ba78576c239f1ea04931979caaf0567 \\\n    --hash=sha256:5daa1f19e097050b9c4d9a78fcc9263cb96c9dfae08037ddc1b7c4ad1889f2a2 \\\n    --hash=sha256:61e9a64c7635095a6bfe483e2ff055d437c59bd45f3617a228b37277f0185d62 \\\n    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \\\n    --hash=sha256:685929988b208a911f1285e2f8ed54210b0d681a3dc0f03e00d599d291986e7e \\\n    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \\\n    --hash=sha256:6b92f60017dda7d877fdc546438b5e28f31c523264f49cf5a48c1d0ce1a0dfbc \\\n    --hash=sha256:70ed9a45c7484d2b30cdacf60d220f494a1763b9fec1ad03285c6553fa0889f2 \\\n    --hash=sha256:719a35fa1156db3640555f95ebb94f60a444e64d1c69626b0edef5df78eba225 \\\n    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \\\n    --hash=sha256:74ea337e0ec3f6f342a36a4f1b5cd94dd9affddcd28ba9aae2905af932ee8c6b \\\n    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \\\n    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \\\n    --hash=sha256:7824b5e8bdbf0bccb4ccd37bbb115849a1dc45437fb4de8351385ed07c437ee0 \\\n    --hash=sha256:7d38b0c279c3032e8c9cc013b405c6df8e1668dbf15465779aa7f15f61201812 \\\n    --hash=sha256:7e9c01d3dd7ceba4d1d436cc021d40d592466e40b9bc7f5d83dc4e98a5c9cd8c \\\n    --hash=sha256:7fd82debf43c6acd0a94359d232f6bb516ee13f269a7993736a9ac9f988bb5d9 \\\n    --hash=sha256:824c3d763a05ea9e9003610145186b0e9848c7584a5575c79bac5a8e7cd80bad \\\n    --hash=sha256:828f75af2b0080c8a972e75f649ab46af008e92c6104a57a759157200b835b75 \\\n    --hash=sha256:83f78128fa28705fa85d01c59771c72fe81c11bd0e6155edbb9f818983a7d761 \\\n    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \\\n    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \\\n    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \\\n    --hash=sha256:8a708a47ade1fe19e8371d5da076bac0dd4b0a5a7985ad6c637f7f7e361b6baa \\\n    --hash=sha256:8af9b142ad719ae3a911ebf616bc4b78b32bbab84d6a40d3ad2f129670509957 \\\n    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \\\n    --hash=sha256:8de6f2a4ce7e7bd27d23dd94abf0ccafe0e0e5cc9c764b0577191f2c25f08f26 \\\n    --hash=sha256:8f8fddb8e323bd6eee4e54e69a39243beab22689070f4c66b472c4cc88bb89d8 \\\n    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \\\n    --hash=sha256:9506e892bcc3b409831d363c6f53e5985e1c8d1f6f6b0256d00358684ff85378 \\\n    --hash=sha256:958254518717542d02d0688d0d20cbf771da5e415e6f49543f92481c850a4540 \\\n    --hash=sha256:95a02752aa032eef4aed01cda6d9b687c669bd0396bf4519eef8bba22a286720 \\\n    --hash=sha256:96c30002424670b5e1e46495c2b8cbffef39cf77c1d79e76462029d50339785b \\\n    --hash=sha256:98b208a7cc42c803445ef551d6753cc42a5ea13e9cab1ee66cd8b9cb70195330 \\\n    --hash=sha256:9b3092d8992a1d69b7a59c3e39f35e1b9be327a17f68a7c35fc17329e337d6f2 \\\n    --hash=sha256:9e51c119992ea8820706871c30a4642ec76de20ae82f9b50b9a45517d8e9f810 \\\n    --hash=sha256:a5716a33bfabb2c6ce27b6cf03253467b3804f83e215f4d202685cf93c6c9874 \\\n    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \\\n    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \\\n    --hash=sha256:a7b85b2cc6ea45e5f7e8c9a30bc9fabd47cda09106cbb4b967335c3e6c43b69d \\\n    --hash=sha256:a83ee7107df13abe42a54a6654670eef9bb39425cf2e27f65e0007465e1286ab \\\n    --hash=sha256:aa7d00b1700966d2917e54d278aba86897890ca9276dd8b76cf6446b6c181b92 \\\n    --hash=sha256:ab620eb663952455271ac37f9aaad86b73c969c02f11f53cea405b38e96a4300 \\\n    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \\\n    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \\\n    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \\\n    --hash=sha256:b390aec180a7c054919c04898835e1c77bced23ea8383eb2c570213bf25d1a86 \\\n    --hash=sha256:b3d78f7bb2b9d9a30345be1474b9aaa8685430b54afb51ba3639b5c6c11e9ed6 \\\n    --hash=sha256:b5664603a253efd3a75716d793d1d3a6a82723b61dc6db767b2460bbbeec4c0f \\\n    --hash=sha256:b69602970994a2ed8bbfa78c2f0394a7435226c6040489702d9f0a0ad0c07052 \\\n    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \\\n    --hash=sha256:bad20d4c69c851c982a1e3606f4c293edfd5a87885786c50082412240c4b1ffd \\\n    --hash=sha256:bb7c99f0673c03017a3ee01e54a5c2617a05468b11eabe513b0080e063ed95b1 \\\n    --hash=sha256:bebb89489b279b2f5661bbbb2abcc87bcd4a46607bb4a5c966f04f1db6b8df9a \\\n    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \\\n    --hash=sha256:c2306e8bb53601979fcb3fa09cc65e031876d9ae01eff2fcbcd7a84ef94d5bc1 \\\n    --hash=sha256:c3a4e41e3096bf1f0f1b76e2ffd6d828d6547f574f702d59bdbef7acfa59db9c \\\n    --hash=sha256:c6834b92dd2428e2dd85ef3d85f723d3c12f20aaf43a2ddd4f944ca25d833408 \\\n    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \\\n    --hash=sha256:ca307d6c259e5c98d3cb9ade55342b47a6839762caf2536f3d7b46ee660cc82e \\\n    --hash=sha256:ca7f6fe0f37ca978a1e5eb7a3a68e6413f417e78e838324947ffd420202b198b \\\n    --hash=sha256:cb6fae641357ed2f6e533c0d3c6504a4a5703621a50c89459e46051d56b61140 \\\n    --hash=sha256:cdaeeb6c350106df6bf9d873395973e5f066a9713200b72cd64f55d0a3eafab6 \\\n    --hash=sha256:cea20da04494e662b83c872683bf4ff2345206043d036315ed0e924b652e7294 \\\n    --hash=sha256:cea90547bfd93807e0013a004dc76552be44fad3bc1cc2b38610a9e889ed098f \\\n    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \\\n    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \\\n    --hash=sha256:d50de98e8d807dc31822fff96f50293163a62418eb65487a21b42713d72ed0b7 \\\n    --hash=sha256:d66a64dd5dec136040ec2ae94aa026a912ee60fdd45bc28d3db30037fd809e88 \\\n    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \\\n    --hash=sha256:da3275833be0edbaf4830fae08bae3dc7219f40ce0c37eaa6c25825957e06612 \\\n    --hash=sha256:dc1a26b8e53395a01c2c611e58602fa47461f136fba7cd5542e6db6d64be1839 \\\n    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \\\n    --hash=sha256:e05c2f7925f1d88778e53cb44f14e0223204a3bdd09a41664750363acfb1f2ef \\\n    --hash=sha256:e12dfea7f5fc2a34a9080efbf79c4c44eb380ec5b9c6fea09407e08f0d1e941d \\\n    --hash=sha256:e4e4523d6f336708d732516e6cfca7796cf3d96c9474eb5aecf6165f2f1fefc3 \\\n    --hash=sha256:e4e49f7e1a4e7191bdf9dc67a974db714501b1fc52c24324103d06a86abd5c08 \\\n    --hash=sha256:e68e151428b5384f766cd25739bf77c7e4a3dc93b5ded7a12118d9fbfdf78ab6 \\\n    --hash=sha256:e8e4d953faaded9ec7ede36824e9814082d22d4c7b1eafbfa079ecba8cd0d076 \\\n    --hash=sha256:ee9df1f0d77b9c6e94f4ac0fec533fbddd5ea3a327807f18d7b069ae019ded80 \\\n    --hash=sha256:f0a887b6565bbfe80efde2b7f6e8890d7d9bbdb11bdb17028a3690c32fe0621f \\\n    --hash=sha256:f0f4a42db92d6ec7677ab9d12830a2a8ec145a9c6d15db2b593466bc875c78d7 \\\n    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \\\n    --hash=sha256:f1d56ec54d257d05e0b50f5780d967540cd07beeaf9e5f645b26d50cce79f4d8 \\\n    --hash=sha256:f4167e87b397f273dc2356fcf1eaf50a6bac51e6105f45103ef7129c8efb0255 \\\n    --hash=sha256:f76fc85bd054c806960f917ec0f329e24e436f1712267d90588e4c39890caa63 \\\n    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a \\\n    --hash=sha256:f9b1c4900736e489a812c529100de4b8fb617d4db075e931e213c57424b83d9b \\\n    --hash=sha256:fc271a6f0a2126958f4090e5507b9da5848927dae331f8f763bd4aa642b3d2cd \\\n    --hash=sha256:febcce10f2bcdbb80b4ea919238a6a4ac13dbc4c7cadbe8d5d75c3682f8b5404\n    # via matplotlib\nmarkupsafe==3.0.3 \\\n    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \\\n    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \\\n    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \\\n    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \\\n    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \\\n    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \\\n    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \\\n    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \\\n    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \\\n    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \\\n    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \\\n    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \\\n    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \\\n    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \\\n    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \\\n    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \\\n    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \\\n    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \\\n    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \\\n    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \\\n    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \\\n    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \\\n    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \\\n    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \\\n    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \\\n    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \\\n    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \\\n    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \\\n    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \\\n    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \\\n    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \\\n    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \\\n    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \\\n    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \\\n    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \\\n    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \\\n    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \\\n    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \\\n    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \\\n    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \\\n    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \\\n    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \\\n    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \\\n    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \\\n    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \\\n    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \\\n    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \\\n    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \\\n    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \\\n    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \\\n    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \\\n    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \\\n    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \\\n    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \\\n    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \\\n    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \\\n    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \\\n    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \\\n    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \\\n    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \\\n    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \\\n    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \\\n    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \\\n    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \\\n    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \\\n    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \\\n    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \\\n    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \\\n    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \\\n    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \\\n    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \\\n    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \\\n    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \\\n    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \\\n    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \\\n    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \\\n    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \\\n    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \\\n    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \\\n    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \\\n    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \\\n    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \\\n    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \\\n    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \\\n    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \\\n    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \\\n    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \\\n    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \\\n    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50\n    # via jinja2\nmatplotlib==3.10.8 \\\n    --hash=sha256:00270d217d6b20d14b584c521f810d60c5c78406dc289859776550df837dcda7 \\\n    --hash=sha256:0a33deb84c15ede243aead39f77e990469fff93ad1521163305095b77b72ce4a \\\n    --hash=sha256:113bb52413ea508ce954a02c10ffd0d565f9c3bc7f2eddc27dfe1731e71c7b5f \\\n    --hash=sha256:12d90df9183093fcd479f4172ac26b322b1248b15729cb57f42f71f24c7e37a3 \\\n    --hash=sha256:15d30132718972c2c074cd14638c7f4592bd98719e2308bccea40e0538bc0cb5 \\\n    --hash=sha256:18821ace09c763ec93aef5eeff087ee493a24051936d7b9ebcad9662f66501f9 \\\n    --hash=sha256:1ae029229a57cd1e8fe542485f27e7ca7b23aa9e8944ddb4985d0bc444f1eca2 \\\n    --hash=sha256:2299372c19d56bcd35cf05a2738308758d32b9eaed2371898d8f5bd33f084aa3 \\\n    --hash=sha256:238b7ce5717600615c895050239ec955d91f321c209dd110db988500558e70d6 \\\n    --hash=sha256:24d50994d8c5816ddc35411e50a86ab05f575e2530c02752e02538122613371f \\\n    --hash=sha256:25d380fe8b1dc32cf8f0b1b448470a77afb195438bafdf1d858bfb876f3edf7b \\\n    --hash=sha256:2c1998e92cd5999e295a731bcb2911c75f597d937341f3030cc24ef2733d78a8 \\\n    --hash=sha256:2cf5bd12cecf46908f286d7838b2abc6c91cda506c0445b8223a7c19a00df008 \\\n    --hash=sha256:32f8dce744be5569bebe789e46727946041199030db8aeb2954d26013a0eb26b \\\n    --hash=sha256:37b3c1cc42aa184b3f738cfa18c1c1d72fd496d85467a6cf7b807936d39aa656 \\\n    --hash=sha256:3a48a78d2786784cc2413e57397981fb45c79e968d99656706018d6e62e57958 \\\n    --hash=sha256:3ab4aabc72de4ff77b3ec33a6d78a68227bf1123465887f9905ba79184a1cc04 \\\n    --hash=sha256:3c624e43ed56313651bc18a47f838b60d7b8032ed348911c54906b130b20071b \\\n    --hash=sha256:3f2e409836d7f5ac2f1c013110a4d50b9f7edc26328c108915f9075d7d7a91b6 \\\n    --hash=sha256:3f5c3e4da343bba819f0234186b9004faba952cc420fbc522dc4e103c1985908 \\\n    --hash=sha256:41703cc95688f2516b480f7f339d8851a6035f18e100ee6a32bc0b8536a12a9c \\\n    --hash=sha256:495672de149445ec1b772ff2c9ede9b769e3cb4f0d0aa7fa730d7f59e2d4e1c1 \\\n    --hash=sha256:4cf267add95b1c88300d96ca837833d4112756045364f5c734a2276038dae27d \\\n    --hash=sha256:56271f3dac49a88d7fca5060f004d9d22b865f743a12a23b1e937a0be4818ee1 \\\n    --hash=sha256:595ba4d8fe983b88f0eec8c26a241e16d6376fe1979086232f481f8f3f67494c \\\n    --hash=sha256:5f62550b9a30afde8c1c3ae450e5eb547d579dd69b25c2fc7a1c67f934c1717a \\\n    --hash=sha256:646d95230efb9ca614a7a594d4fcacde0ac61d25e37dd51710b36477594963ce \\\n    --hash=sha256:64fcc24778ca0404ce0cb7b6b77ae1f4c7231cdd60e6778f999ee05cbd581b9a \\\n    --hash=sha256:6be43b667360fef5c754dda5d25a32e6307a03c204f3c0fc5468b78fa87b4160 \\\n    --hash=sha256:6da7c2ce169267d0d066adcf63758f0604aa6c3eebf67458930f9d9b79ad1db1 \\\n    --hash=sha256:83d282364ea9f3e52363da262ce32a09dfe241e4080dcedda3c0db059d3c1f11 \\\n    --hash=sha256:9153c3292705be9f9c64498a8872118540c3f4123d1a1c840172edf262c8be4a \\\n    --hash=sha256:99eefd13c0dc3b3c1b4d561c1169e65fe47aab7b8158754d7c084088e2329466 \\\n    --hash=sha256:a0a7f52498f72f13d4a25ea70f35f4cb60642b466cbb0a9be951b5bc3f45a486 \\\n    --hash=sha256:a2b336e2d91a3d7006864e0990c83b216fcdca64b5a6484912902cef87313d78 \\\n    --hash=sha256:a48f2b74020919552ea25d222d5cc6af9ca3f4eb43a93e14d068457f545c2a17 \\\n    --hash=sha256:ad3d9833a64cf48cc4300f2b406c3d0f4f4724a91c0bd5640678a6ba7c102077 \\\n    --hash=sha256:b44d07310e404ba95f8c25aa5536f154c0a8ec473303535949e52eb71d0a1565 \\\n    --hash=sha256:b53285e65d4fa4c86399979e956235deb900be5baa7fc1218ea67fbfaeaadd6f \\\n    --hash=sha256:b5a2b97dbdc7d4f353ebf343744f1d1f1cca8aa8bfddb4262fcf4306c3761d50 \\\n    --hash=sha256:b9a5ca4ac220a0cdd1ba6bcba3608547117d30468fefce49bb26f55c1a3d5c58 \\\n    --hash=sha256:bab485bcf8b1c7d2060b4fcb6fc368a9e6f4cd754c9c2fea281f4be21df394a2 \\\n    --hash=sha256:c108a1d6fa78a50646029cb6d49808ff0fc1330fda87fa6f6250c6b5369b6645 \\\n    --hash=sha256:d56a1efd5bfd61486c8bc968fa18734464556f0fb8e51690f4ac25d85cbbbbc2 \\\n    --hash=sha256:d9050fee89a89ed57b4fb2c1bfac9a3d0c57a0d55aed95949eedbc42070fea39 \\\n    --hash=sha256:dd80ecb295460a5d9d260df63c43f4afbdd832d725a531f008dad1664f458adf \\\n    --hash=sha256:e8ea3e2d4066083e264e75c829078f9e149fa119d27e19acd503de65e0b13149 \\\n    --hash=sha256:eb3823f11823deade26ce3b9f40dcb4a213da7a670013929f31d5f5ed1055b22 \\\n    --hash=sha256:ee40c27c795bda6a5292e9cff9890189d32f7e3a0bf04e0e3c9430c4a00c37df \\\n    --hash=sha256:efb30e3baaea72ce5928e32bab719ab4770099079d66726a62b11b1ef7273be4 \\\n    --hash=sha256:f254d118d14a7f99d616271d6c3c27922c092dac11112670b157798b89bf4933 \\\n    --hash=sha256:f89c151aab2e2e23cb3fe0acad1e8b82841fd265379c4cecd0f3fcb34c15e0f6 \\\n    --hash=sha256:f97aeb209c3d2511443f8797e3e5a569aebb040d4f8bc79aa3ee78a8fb9e3dd8 \\\n    --hash=sha256:f9b587c9c7274c1613a30afabf65a272114cd6cdbe67b3406f818c79d7ab2e2a \\\n    --hash=sha256:fb061f596dad3a0f52b60dc6a5dec4a0c300dec41e058a7efe09256188d170b7\n    # via -r tools/reef-requirements.in\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nnumpy==2.3.5 \\\n    --hash=sha256:00dc4e846108a382c5869e77c6ed514394bdeb3403461d25a829711041217d5b \\\n    --hash=sha256:0472f11f6ec23a74a906a00b48a4dcf3849209696dff7c189714511268d103ae \\\n    --hash=sha256:04822c00b5fd0323c8166d66c701dc31b7fbd252c100acd708c48f763968d6a3 \\\n    --hash=sha256:052e8c42e0c49d2575621c158934920524f6c5da05a1d3b9bab5d8e259e045f0 \\\n    --hash=sha256:09a1bea522b25109bf8e6f3027bd810f7c1085c64a0c7ce050c1676ad0ba010b \\\n    --hash=sha256:0cd00b7b36e35398fa2d16af7b907b65304ef8bb4817a550e06e5012929830fa \\\n    --hash=sha256:0d8163f43acde9a73c2a33605353a4f1bc4798745a8b1d73183b28e5b435ae28 \\\n    --hash=sha256:1062fde1dcf469571705945b0f221b73928f34a20c904ffb45db101907c3454e \\\n    --hash=sha256:11e06aa0af8c0f05104d56450d6093ee639e15f24ecf62d417329d06e522e017 \\\n    --hash=sha256:17531366a2e3a9e30762c000f2c43a9aaa05728712e25c11ce1dbe700c53ad41 \\\n    --hash=sha256:1978155dd49972084bd6ef388d66ab70f0c323ddee6f693d539376498720fb7e \\\n    --hash=sha256:1ed1ec893cff7040a02c8aa1c8611b94d395590d553f6b53629a4461dc7f7b63 \\\n    --hash=sha256:2dcd0808a421a482a080f89859a18beb0b3d1e905b81e617a188bd80422d62e9 \\\n    --hash=sha256:2e2eb32ddb9ccb817d620ac1d8dae7c3f641c1e5f55f531a33e8ab97960a75b8 \\\n    --hash=sha256:2feae0d2c91d46e59fcd62784a3a83b3fb677fead592ce51b5a6fbb4f95965ff \\\n    --hash=sha256:3095bdb8dd297e5920b010e96134ed91d852d81d490e787beca7e35ae1d89cf7 \\\n    --hash=sha256:30bc11310e8153ca664b14c5f1b73e94bd0503681fcf136a163de856f3a50139 \\\n    --hash=sha256:3101e5177d114a593d79dd79658650fe28b5a0d8abeb8ce6f437c0e6df5be1a4 \\\n    --hash=sha256:396084a36abdb603546b119d96528c2f6263921c50df3c8fd7cb28873a237748 \\\n    --hash=sha256:3997b5b3c9a771e157f9aae01dd579ee35ad7109be18db0e85dbdbe1de06e952 \\\n    --hash=sha256:414802f3b97f3c1eef41e530aaba3b3c1620649871d8cb38c6eaff034c2e16bd \\\n    --hash=sha256:51c1e14eb1e154ebd80e860722f9e6ed6ec89714ad2db2d3aa33c31d7c12179b \\\n    --hash=sha256:51c55fe3451421f3a6ef9a9c1439e82101c57a2c9eab9feb196a62b1a10b58ce \\\n    --hash=sha256:5ee6609ac3604fa7780e30a03e5e241a7956f8e2fcfe547d51e3afa5247ac47f \\\n    --hash=sha256:612a95a17655e213502f60cfb9bf9408efdc9eb1d5f50535cc6eb365d11b42b5 \\\n    --hash=sha256:6203fdf9f3dc5bdaed7319ad8698e685c7a3be10819f41d32a0723e611733b42 \\\n    --hash=sha256:63c0e9e7eea69588479ebf4a8a270d5ac22763cc5854e9a7eae952a3908103f7 \\\n    --hash=sha256:66f85ce62c70b843bab1fb14a05d5737741e74e28c7b8b5a064de10142fad248 \\\n    --hash=sha256:6cf9b429b21df6b99f4dee7a1218b8b7ffbbe7df8764dc0bd60ce8a0708fed1e \\\n    --hash=sha256:70b37199913c1bd300ff6e2693316c6f869c7ee16378faf10e4f5e3275b299c3 \\\n    --hash=sha256:727fd05b57df37dc0bcf1a27767a3d9a78cbbc92822445f32cc3436ba797337b \\\n    --hash=sha256:74ae7b798248fe62021dbf3c914245ad45d1a6b0cb4a29ecb4b31d0bfbc4cc3e \\\n    --hash=sha256:784db1dcdab56bf0517743e746dfb0f885fc68d948aba86eeec2cba234bdf1c0 \\\n    --hash=sha256:86945f2ee6d10cdfd67bcb4069c1662dd711f7e2a4343db5cecec06b87cf31aa \\\n    --hash=sha256:86d835afea1eaa143012a2d7a3f45a3adce2d7adc8b4961f0b362214d800846a \\\n    --hash=sha256:872a5cf366aec6bb1147336480fef14c9164b154aeb6542327de4970282cd2f5 \\\n    --hash=sha256:8b973c57ff8e184109db042c842423ff4f60446239bd585a5131cc47f06f789d \\\n    --hash=sha256:8cba086a43d54ca804ce711b2a940b16e452807acebe7852ff327f1ecd49b0d4 \\\n    --hash=sha256:8f7f0e05112916223d3f438f293abf0727e1181b5983f413dfa2fefc4098245c \\\n    --hash=sha256:900218e456384ea676e24ea6a0417f030a3b07306d29d7ad843957b40a9d8d52 \\\n    --hash=sha256:93eebbcf1aafdf7e2ddd44c2923e2672e1010bddc014138b229e49725b4d6be5 \\\n    --hash=sha256:9c75442b2209b8470d6d5d8b1c25714270686f14c749028d2199c54e29f20b4d \\\n    --hash=sha256:9ee2197ef8c4f0dfe405d835f3b6a14f5fee7782b5de51ba06fb65fc9b36e9f1 \\\n    --hash=sha256:a414504bef8945eae5f2d7cb7be2d4af77c5d1cb5e20b296c2c25b61dff2900c \\\n    --hash=sha256:a4b9159734b326535f4dd01d947f919c6eefd2d9827466a696c44ced82dfbc18 \\\n    --hash=sha256:a80afd79f45f3c4a7d341f13acbe058d1ca8ac017c165d3fa0d3de6bc1a079d7 \\\n    --hash=sha256:aa5bc7c5d59d831d9773d1170acac7893ce3a5e130540605770ade83280e7188 \\\n    --hash=sha256:acfd89508504a19ed06ef963ad544ec6664518c863436306153e13e94605c218 \\\n    --hash=sha256:aeffcab3d4b43712bb7a60b65f6044d444e75e563ff6180af8f98dd4b905dfd2 \\\n    --hash=sha256:afaffc4393205524af9dfa400fa250143a6c3bc646c08c9f5e25a9f4b4d6a903 \\\n    --hash=sha256:b0c7088a73aef3d687c4deef8452a3ac7c1be4e29ed8bf3b366c8111128ac60c \\\n    --hash=sha256:b46b4ec24f7293f23adcd2d146960559aaf8020213de8ad1909dba6c013bf89c \\\n    --hash=sha256:b501b5fa195cc9e24fe102f21ec0a44dffc231d2af79950b451e0d99cea02234 \\\n    --hash=sha256:bf06bc2af43fa8d32d30fae16ad965663e966b1a3202ed407b84c989c3221e82 \\\n    --hash=sha256:c804e3a5aba5460c73955c955bdbd5c08c354954e9270a2c1565f62e866bdc39 \\\n    --hash=sha256:c8a9958e88b65c3b27e22ca2a076311636850b612d6bbfb76e8d156aacde2aaf \\\n    --hash=sha256:cc0a57f895b96ec78969c34f682c602bf8da1a0270b09bc65673df2e7638ec20 \\\n    --hash=sha256:cc8920d2ec5fa99875b670bb86ddeb21e295cb07aa331810d9e486e0b969d946 \\\n    --hash=sha256:ccc933afd4d20aad3c00bcef049cb40049f7f196e0397f1109dba6fed63267b0 \\\n    --hash=sha256:ce581db493ea1a96c0556360ede6607496e8bf9b3a8efa66e06477267bc831e9 \\\n    --hash=sha256:d0f23b44f57077c1ede8c5f26b30f706498b4862d3ff0a7298b8411dd2f043ff \\\n    --hash=sha256:d21644de1b609825ede2f48be98dfde4656aefc713654eeee280e37cadc4e0ad \\\n    --hash=sha256:d6889ec4ec662a1a37eb4b4fb26b6100841804dac55bd9df579e326cdc146227 \\\n    --hash=sha256:de5672f4a7b200c15a4127042170a694d4df43c992948f5e1af57f0174beed10 \\\n    --hash=sha256:e6a0bc88393d65807d751a614207b7129a310ca4fe76a74e5c7da5fa5671417e \\\n    --hash=sha256:ed89927b86296067b4f81f108a2271d8926467a8868e554eaf370fc27fa3ccaf \\\n    --hash=sha256:ee3888d9ff7c14604052b2ca5535a30216aa0a58e948cdd3eeb8d3415f638769 \\\n    --hash=sha256:f0963b55cdd70fad460fa4c1341f12f976bb26cb66021a5580329bd498988310 \\\n    --hash=sha256:f16417ec91f12f814b10bafe79ef77e70113a2f5f7018640e7425ff979253425 \\\n    --hash=sha256:f28620fe26bee16243be2b7b874da327312240a7cdc38b769a697578d2100013 \\\n    --hash=sha256:f4255143f5160d0de972d28c8f9665d882b5f61309d8362fdd3e103cf7bf010c \\\n    --hash=sha256:ffac52f28a7849ad7576293c0cb7b9f08304e8f7d738a8cb8a90ec4c55a998eb \\\n    --hash=sha256:ffe22d2b05504f786c867c8395de703937f934272eb67586817b46188b4ded6d \\\n    --hash=sha256:fffe29a1ef00883599d1dc2c51aa2e5d80afe49523c261a74933df395c15c520\n    # via\n    #   -r tools/reef-requirements.in\n    #   accelerate\n    #   chronos-forecasting\n    #   contourpy\n    #   matplotlib\n    #   pandas\n    #   transformers\nnvidia-cublas-cu12==12.8.4.1 \\\n    --hash=sha256:47e9b82132fa8d2b4944e708049229601448aaad7e6f296f630f2d1a32de35af \\\n    --hash=sha256:8ac4e771d5a348c551b2a426eda6193c19aa630236b418086020df5ba9667142 \\\n    --hash=sha256:b86f6dd8935884615a0683b663891d43781b819ac4f2ba2b0c9604676af346d0\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90 \\\n    --hash=sha256:4412396548808ddfed3f17a467b104ba7751e6b58678a4b840675c56d21cf7ed \\\n    --hash=sha256:bb479dcdf7e6d4f8b0b01b115260399bf34154a1a2e9fe11c85c517d87efd98e \\\n    --hash=sha256:ea0cb07ebda26bb9b29ba82cda34849e73c166c18162d3913575b0c9db9a6182\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93 \\\n    --hash=sha256:7a4b6b2904850fe78e0bd179c4b655c404d4bb799ef03ddc60804247099ae909 \\\n    --hash=sha256:a7756528852ef889772a84c6cd89d41dfa74667e24cca16bb31f8f061e3e9994 \\\n    --hash=sha256:fc1fec1e1637854b4c0a65fb9a8346b51dd9ee69e61ebaccc82058441f15bce8\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90 \\\n    --hash=sha256:52bf7bbee900262ffefe5e9d5a2a69a30d97e2bc5bb6cc866688caa976966e3d \\\n    --hash=sha256:adade8dcbd0edf427b7204d480d6066d33902cab2a4707dcfc48a2d0fd44ab90 \\\n    --hash=sha256:c0c6027f01505bfed6c3b21ec546f69c687689aad5f1a377554bc6ca4aa993a8\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21 \\\n    --hash=sha256:949452be657fa16687d0930933f032835951ef0892b37d2d53824d1a84dc97a8 \\\n    --hash=sha256:c6288de7d63e6cf62988f0923f96dc339cea362decb1bf5b3141883392a7d65e \\\n    --hash=sha256:c9132cc3f8958447b4910a1720036d9eff5928cc3179b0a51fb6d167c6cc87d8\n    # via torch\nnvidia-cufft-cu12==11.3.3.83 \\\n    --hash=sha256:4d2dd21ec0b88cf61b62e6b43564355e5222e4a3fb394cac0db101f2dd0d4f74 \\\n    --hash=sha256:7a64a98ef2a7c47f905aaf8931b69a3a43f27c55530c698bb2ed7c75c0b42cb7 \\\n    --hash=sha256:848ef7224d6305cdb2a4df928759dca7b1201874787083b6e7550dd6765ce69a\n    # via torch\nnvidia-cufile-cu12==1.13.1.3 \\\n    --hash=sha256:1d069003be650e131b21c932ec3d8969c1715379251f8d23a1860554b1cb24fc \\\n    --hash=sha256:4beb6d4cce47c1a0f1013d72e02b0994730359e17801d395bdcbf20cfb3bb00a\n    # via torch\nnvidia-curand-cu12==10.3.9.90 \\\n    --hash=sha256:b32331d4f4df5d6eefa0554c565b626c7216f87a06a4f56fab27c3b68a830ec9 \\\n    --hash=sha256:dfab99248034673b779bc6decafdc3404a8a6f502462201f2f31f11354204acd \\\n    --hash=sha256:f149a8ca457277da854f89cf282d6ef43176861926c7ac85b2a0fbd237c587ec\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90 \\\n    --hash=sha256:4376c11ad263152bd50ea295c05370360776f8c3427b30991df774f9fb26c450 \\\n    --hash=sha256:4a550db115fcabc4d495eb7d39ac8b58d4ab5d8e63274d3754df1c0ad6a22d34 \\\n    --hash=sha256:db9ed69dbef9715071232caa9b69c52ac7de3a95773c2db65bdba85916e4e5c0\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93 \\\n    --hash=sha256:1ec05d76bbbd8b61b06a80e1eaf8cf4959c3d4ce8e711b65ebd0443bb0ebb13b \\\n    --hash=sha256:9a33604331cb2cac199f2e7f5104dfbb8a5a898c367a53dfda9ff2acb6b6b4dd \\\n    --hash=sha256:9b6c161cb130be1a07a27ea6923df8141f3c295852f4b260c65f18f3e0a091dc\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1 \\\n    --hash=sha256:8878dce784d0fac90131b6817b607e803c36e629ba34dc5b433471382196b6a5 \\\n    --hash=sha256:f1bb701d6b930d5a7cea44c19ceb973311500847f81b634d802b7b539dc55623 \\\n    --hash=sha256:f67fbb5831940ec829c9117b7f33807db9f9678dc2a617fbe781cac17b4e1075\n    # via torch\nnvidia-nccl-cu12==2.27.5 \\\n    --hash=sha256:31432ad4d1fb1004eb0c56203dc9bc2178a1ba69d1d9e02d64a6938ab5e40e7a \\\n    --hash=sha256:ad730cf15cb5d25fe849c6e6ca9eb5b76db16a80f13f425ac68d8e2e55624457\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93 \\\n    --hash=sha256:81ff63371a7ebd6e6451970684f916be2eab07321b73c9d244dc2b4da7f73b88 \\\n    --hash=sha256:adccd7161ace7261e01bb91e44e88da350895c270d23f744f0820c818b7229e7 \\\n    --hash=sha256:bd93fbeeee850917903583587f4fc3a4eafa022e34572251368238ab5e6bd67f\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvshmem-cu12==3.3.20 \\\n    --hash=sha256:0b0b960da3842212758e4fa4696b94f129090b30e5122fea3c5345916545cff0 \\\n    --hash=sha256:d00f26d3f9b2e3c3065be895e3059d6479ea5c638a3f38c9fec49b1b9dd7c1e5\n    # via torch\nnvidia-nvtx-cu12==12.8.90 \\\n    --hash=sha256:5b17e2001cc0d751a5bc2c6ec6d26ad95913324a4adb86788c944f8ce9ba441f \\\n    --hash=sha256:619c8304aedc69f02ea82dd244541a83c3d9d40993381b3b590f1adaed3db41e \\\n    --hash=sha256:d7ad891da111ebafbf7e015d34879f7112832fc239ff0d7d776b6cb685274615\n    # via torch\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   accelerate\n    #   huggingface-hub\n    #   matplotlib\n    #   transformers\npandas==2.3.3 \\\n    --hash=sha256:0242fe9a49aa8b4d78a4fa03acb397a58833ef6199e9aa40a95f027bb3a1b6e7 \\\n    --hash=sha256:1611aedd912e1ff81ff41c745822980c49ce4a7907537be8692c8dbc31924593 \\\n    --hash=sha256:1b07204a219b3b7350abaae088f451860223a52cfb8a6c53358e7948735158e5 \\\n    --hash=sha256:1d37b5848ba49824e5c30bedb9c830ab9b7751fd049bc7914533e01c65f79791 \\\n    --hash=sha256:23ebd657a4d38268c7dfbdf089fbc31ea709d82e4923c5ffd4fbd5747133ce73 \\\n    --hash=sha256:2462b1a365b6109d275250baaae7b760fd25c726aaca0054649286bcfbb3e8ec \\\n    --hash=sha256:28083c648d9a99a5dd035ec125d42439c6c1c525098c58af0fc38dd1a7a1b3d4 \\\n    --hash=sha256:2e3ebdb170b5ef78f19bfb71b0dc5dc58775032361fa188e814959b74d726dd5 \\\n    --hash=sha256:318d77e0e42a628c04dc56bcef4b40de67918f7041c2b061af1da41dcff670ac \\\n    --hash=sha256:371a4ab48e950033bcf52b6527eccb564f52dc826c02afd9a1bc0ab731bba084 \\\n    --hash=sha256:376c6446ae31770764215a6c937f72d917f214b43560603cd60da6408f183b6c \\\n    --hash=sha256:3869faf4bd07b3b66a9f462417d0ca3a9df29a9f6abd5d0d0dbab15dac7abe87 \\\n    --hash=sha256:3fd2f887589c7aa868e02632612ba39acb0b8948faf5cc58f0850e165bd46f35 \\\n    --hash=sha256:4793891684806ae50d1288c9bae9330293ab4e083ccd1c5e383c34549c6e4250 \\\n    --hash=sha256:4e0a175408804d566144e170d0476b15d78458795bb18f1304fb94160cabf40c \\\n    --hash=sha256:503cf027cf9940d2ceaa1a93cfb5f8c8c7e6e90720a2850378f0b3f3b1e06826 \\\n    --hash=sha256:5554c929ccc317d41a5e3d1234f3be588248e61f08a74dd17c9eabb535777dc9 \\\n    --hash=sha256:56851a737e3470de7fa88e6131f41281ed440d29a9268dcbf0002da5ac366713 \\\n    --hash=sha256:5caf26f64126b6c7aec964f74266f435afef1c1b13da3b0636c7518a1fa3e2b1 \\\n    --hash=sha256:602b8615ebcc4a0c1751e71840428ddebeb142ec02c786e8ad6b1ce3c8dec523 \\\n    --hash=sha256:6253c72c6a1d990a410bc7de641d34053364ef8bcd3126f7e7450125887dffe3 \\\n    --hash=sha256:6435cb949cb34ec11cc9860246ccb2fdc9ecd742c12d3304989017d53f039a78 \\\n    --hash=sha256:6d21f6d74eb1725c2efaa71a2bfc661a0689579b58e9c0ca58a739ff0b002b53 \\\n    --hash=sha256:6d2cefc361461662ac48810cb14365a365ce864afe85ef1f447ff5a1e99ea81c \\\n    --hash=sha256:74ecdf1d301e812db96a465a525952f4dde225fdb6d8e5a521d47e1f42041e21 \\\n    --hash=sha256:75ea25f9529fdec2d2e93a42c523962261e567d250b0013b16210e1d40d7c2e5 \\\n    --hash=sha256:854d00d556406bffe66a4c0802f334c9ad5a96b4f1f868adf036a21b11ef13ff \\\n    --hash=sha256:8fe25fc7b623b0ef6b5009149627e34d2a4657e880948ec3c840e9402e5c1b45 \\\n    --hash=sha256:900f47d8f20860de523a1ac881c4c36d65efcb2eb850e6948140fa781736e110 \\\n    --hash=sha256:93c2d9ab0fc11822b5eece72ec9587e172f63cff87c00b062f6e37448ced4493 \\\n    --hash=sha256:a16dcec078a01eeef8ee61bf64074b4e524a2a3f4b3be9326420cabe59c4778b \\\n    --hash=sha256:a21d830e78df0a515db2b3d2f5570610f5e6bd2e27749770e8bb7b524b89b450 \\\n    --hash=sha256:a45c765238e2ed7d7c608fc5bc4a6f88b642f2f01e70c0c23d2224dd21829d86 \\\n    --hash=sha256:a637c5cdfa04b6d6e2ecedcb81fc52ffb0fd78ce2ebccc9ea964df9f658de8c8 \\\n    --hash=sha256:a68e15f780eddf2b07d242e17a04aa187a7ee12b40b930bfdd78070556550e98 \\\n    --hash=sha256:b3d11d2fda7eb164ef27ffc14b4fcab16a80e1ce67e9f57e19ec0afaf715ba89 \\\n    --hash=sha256:b468d3dad6ff947df92dcb32ede5b7bd41a9b3cceef0a30ed925f6d01fb8fa66 \\\n    --hash=sha256:b98560e98cb334799c0b07ca7967ac361a47326e9b4e5a7dfb5ab2b1c9d35a1b \\\n    --hash=sha256:bdcd9d1167f4885211e401b3036c0c8d9e274eee67ea8d0758a256d60704cfe8 \\\n    --hash=sha256:bf1f8a81d04ca90e32a0aceb819d34dbd378a98bf923b6398b9a3ec0bf44de29 \\\n    --hash=sha256:c46467899aaa4da076d5abc11084634e2d197e9460643dd455ac3db5856b24d6 \\\n    --hash=sha256:c4fc4c21971a1a9f4bdb4c73978c7f7256caa3e62b323f70d6cb80db583350bc \\\n    --hash=sha256:c503ba5216814e295f40711470446bc3fd00f0faea8a086cbc688808e26f92a2 \\\n    --hash=sha256:d051c0e065b94b7a3cea50eb1ec32e912cd96dba41647eb24104b6c6c14c5788 \\\n    --hash=sha256:d3e28b3e83862ccf4d85ff19cf8c20b2ae7e503881711ff2d534dc8f761131aa \\\n    --hash=sha256:db4301b2d1f926ae677a751eb2bd0e8c5f5319c9cb3f88b0becbbb0b07b34151 \\\n    --hash=sha256:dd7478f1463441ae4ca7308a70e90b33470fa593429f9d4c578dd00d1fa78838 \\\n    --hash=sha256:e05e1af93b977f7eafa636d043f9f94c7ee3ac81af99c13508215942e64c993b \\\n    --hash=sha256:e19d192383eab2f4ceb30b412b22ea30690c9e618f78870357ae1d682912015a \\\n    --hash=sha256:e32e7cc9af0f1cc15548288a51a3b681cc2a219faa838e995f7dc53dbab1062d \\\n    --hash=sha256:ecaf1e12bdc03c86ad4a7ea848d66c685cb6851d807a26aa245ca3d2017a1908 \\\n    --hash=sha256:ee15f284898e7b246df8087fc82b87b01686f98ee67d85a17b7ab44143a3a9a0 \\\n    --hash=sha256:ee67acbbf05014ea6c763beb097e03cd629961c8a632075eeb34247120abcb4b \\\n    --hash=sha256:f086f6fe114e19d92014a1966f43a3e62285109afe874f067f5abbdcbb10e59c \\\n    --hash=sha256:f8bfc0e12dc78f777f323f55c58649591b2cd0c43534e8355c51d3fede5f4dee\n    # via\n    #   -r tools/reef-requirements.in\n    #   chronos-forecasting\npillow==12.3.0 \\\n    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \\\n    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \\\n    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \\\n    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \\\n    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \\\n    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \\\n    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \\\n    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \\\n    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \\\n    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \\\n    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \\\n    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \\\n    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \\\n    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \\\n    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \\\n    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \\\n    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \\\n    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \\\n    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \\\n    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \\\n    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \\\n    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \\\n    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \\\n    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \\\n    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \\\n    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \\\n    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \\\n    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \\\n    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \\\n    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \\\n    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \\\n    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \\\n    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \\\n    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \\\n    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \\\n    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \\\n    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \\\n    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \\\n    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \\\n    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \\\n    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \\\n    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \\\n    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \\\n    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \\\n    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \\\n    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \\\n    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \\\n    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \\\n    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \\\n    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \\\n    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \\\n    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \\\n    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \\\n    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \\\n    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \\\n    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \\\n    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \\\n    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \\\n    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \\\n    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \\\n    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \\\n    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \\\n    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \\\n    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \\\n    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \\\n    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \\\n    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \\\n    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \\\n    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \\\n    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \\\n    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \\\n    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \\\n    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \\\n    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \\\n    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \\\n    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \\\n    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \\\n    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \\\n    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \\\n    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \\\n    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \\\n    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \\\n    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \\\n    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \\\n    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \\\n    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \\\n    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7\n    # via matplotlib\npsutil==7.2.2 \\\n    --hash=sha256:0746f5f8d406af344fd547f1c8daa5f5c33dbc293bb8d6a16d80b4bb88f59372 \\\n    --hash=sha256:076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9 \\\n    --hash=sha256:11fe5a4f613759764e79c65cf11ebdf26e33d6dd34336f8a337aa2996d71c841 \\\n    --hash=sha256:1a571f2330c966c62aeda00dd24620425d4b0cc86881c89861fbc04549e5dc63 \\\n    --hash=sha256:1a7b04c10f32cc88ab39cbf606e117fd74721c831c98a27dc04578deb0c16979 \\\n    --hash=sha256:1fa4ecf83bcdf6e6c8f4449aff98eefb5d0604bf88cb883d7da3d8d2d909546a \\\n    --hash=sha256:2edccc433cbfa046b980b0df0171cd25bcaeb3a68fe9022db0979e7aa74a826b \\\n    --hash=sha256:7b6d09433a10592ce39b13d7be5a54fbac1d1228ed29abc880fb23df7cb694c9 \\\n    --hash=sha256:8c233660f575a5a89e6d4cb65d9f938126312bca76d8fe087b947b3a1aaac9ee \\\n    --hash=sha256:917e891983ca3c1887b4ef36447b1e0873e70c933afc831c6b6da078ba474312 \\\n    --hash=sha256:ab486563df44c17f5173621c7b198955bd6b613fb87c71c161f827d3fb149a9b \\\n    --hash=sha256:ae0aefdd8796a7737eccea863f80f81e468a1e4cf14d926bd9b6f5f2d5f90ca9 \\\n    --hash=sha256:b0726cecd84f9474419d67252add4ac0cd9811b04d61123054b9fb6f57df6e9e \\\n    --hash=sha256:b58fabe35e80b264a4e3bb23e6b96f9e45a3df7fb7eed419ac0e5947c61e47cc \\\n    --hash=sha256:c7663d4e37f13e884d13994247449e9f8f574bc4655d509c3b95e9ec9e2b9dc1 \\\n    --hash=sha256:e452c464a02e7dc7822a05d25db4cde564444a67e58539a00f929c51eddda0cf \\\n    --hash=sha256:e78c8603dcd9a04c7364f1a3e670cea95d51ee865e4efb3556a3a63adef958ea \\\n    --hash=sha256:eb7e81434c8d223ec4a219b5fc1c47d0417b12be7ea866e24fb5ad6e84b3d988 \\\n    --hash=sha256:ed0cace939114f62738d808fdcecd4c869222507e266e574799e9c0faa17d486 \\\n    --hash=sha256:eed63d3b4d62449571547b60578c5b2c4bcccc5387148db46e0c2313dad0ee00 \\\n    --hash=sha256:fd04ef36b4a6d599bbdb225dd1d3f51e00105f6d48a28f006da7f9822f2606d8\n    # via accelerate\npyparsing==3.3.3 \\\n    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \\\n    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4\n    # via matplotlib\npython-dateutil==2.9.0.post0 \\\n    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \\\n    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\n    # via\n    #   matplotlib\n    #   pandas\npytz==2026.4 \\\n    --hash=sha256:464303645bafafd72418898368b2429458f709cf1eb6a15372fbcc396b64da63 \\\n    --hash=sha256:9d514388fbc89ca0833203464272ac485b8828568ab73532f5020f17e892a0ff\n    # via pandas\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   accelerate\n    #   huggingface-hub\n    #   transformers\nregex==2026.9.10 \\\n    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \\\n    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \\\n    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \\\n    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \\\n    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \\\n    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \\\n    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \\\n    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \\\n    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \\\n    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \\\n    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \\\n    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \\\n    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \\\n    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \\\n    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \\\n    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \\\n    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \\\n    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \\\n    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \\\n    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \\\n    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \\\n    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \\\n    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \\\n    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \\\n    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \\\n    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \\\n    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \\\n    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \\\n    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \\\n    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \\\n    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \\\n    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \\\n    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \\\n    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \\\n    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \\\n    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \\\n    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \\\n    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \\\n    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \\\n    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \\\n    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \\\n    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \\\n    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \\\n    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \\\n    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \\\n    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \\\n    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \\\n    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \\\n    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \\\n    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \\\n    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \\\n    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \\\n    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \\\n    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \\\n    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \\\n    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \\\n    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \\\n    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \\\n    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \\\n    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \\\n    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \\\n    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \\\n    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \\\n    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \\\n    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \\\n    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \\\n    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \\\n    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \\\n    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \\\n    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \\\n    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \\\n    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \\\n    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \\\n    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \\\n    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \\\n    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \\\n    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \\\n    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \\\n    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \\\n    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \\\n    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \\\n    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \\\n    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \\\n    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \\\n    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \\\n    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \\\n    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \\\n    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \\\n    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \\\n    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \\\n    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \\\n    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \\\n    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \\\n    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \\\n    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \\\n    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \\\n    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \\\n    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \\\n    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \\\n    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \\\n    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \\\n    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \\\n    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \\\n    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \\\n    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \\\n    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \\\n    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \\\n    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \\\n    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \\\n    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \\\n    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \\\n    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \\\n    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \\\n    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \\\n    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \\\n    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \\\n    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \\\n    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \\\n    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \\\n    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \\\n    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \\\n    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \\\n    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \\\n    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \\\n    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \\\n    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \\\n    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \\\n    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \\\n    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \\\n    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07\n    # via transformers\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via\n    #   huggingface-hub\n    #   transformers\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via\n    #   -r tools/reef-requirements.in\n    #   accelerate\n    #   transformers\nsetuptools==84.0.0 \\\n    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n    # via torch\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\n    # via python-dateutil\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\ntokenizers==0.22.2 \\\n    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n    # via transformers\ntorch==2.9.1 \\\n    --hash=sha256:07c8a9660bc9414c39cac530ac83b1fb1b679d7155824144a40a54f4a47bfa73 \\\n    --hash=sha256:0a2bd769944991c74acf0c4ef23603b9c777fdf7637f115605a4b2d8023110c7 \\\n    --hash=sha256:0d06b30a9207b7c3516a9e0102114024755a07045f0c1d2f2a56b1819ac06bcb \\\n    --hash=sha256:19d144d6b3e29921f1fc70503e9f2fc572cde6a5115c0c0de2f7ca8b1483e8b6 \\\n    --hash=sha256:1cc208435f6c379f9b8fdfd5ceb5be1e3b72a6bdf1cb46c0d2812aa73472db9e \\\n    --hash=sha256:1edee27a7c9897f4e0b7c14cfc2f3008c571921134522d5b9b5ec4ebbc69041a \\\n    --hash=sha256:27331cd902fb4322252657f3902adf1c4f6acad9dcad81d8df3ae14c7c4f07c4 \\\n    --hash=sha256:2af70e3be4a13becba4655d6cc07dcfec7ae844db6ac38d6c1dafeb245d17d65 \\\n    --hash=sha256:2c14b3da5df416cf9cb5efab83aa3056f5b8cd8620b8fde81b4987ecab730587 \\\n    --hash=sha256:2e1c42c0ae92bf803a4b2409fdfed85e30f9027a66887f5e7dcdbc014c7531db \\\n    --hash=sha256:30a3e170a84894f3652434b56d59a64a2c11366b0ed5776fab33c2439396bf9a \\\n    --hash=sha256:52347912d868653e1528b47cafaf79b285b98be3f4f35d5955389b1b95224475 \\\n    --hash=sha256:524de44cd13931208ba2c4bde9ec7741fd4ae6bfd06409a604fc32f6520c2bc9 \\\n    --hash=sha256:545844cc16b3f91e08ce3b40e9c2d77012dd33a48d505aed34b7740ed627a1b2 \\\n    --hash=sha256:5be4bf7496f1e3ffb1dd44b672adb1ac3f081f204c5ca81eba6442f5f634df8e \\\n    --hash=sha256:62b3fd888277946918cba4478cf849303da5359f0fb4e3bfb86b0533ba2eaf8d \\\n    --hash=sha256:81a285002d7b8cfd3fdf1b98aa8df138d41f1a8334fd9ea37511517cedf43083 \\\n    --hash=sha256:8301a7b431e51764629208d0edaa4f9e4c33e6df0f2f90b90e261d623df6a4e2 \\\n    --hash=sha256:9fd35c68b3679378c11f5eb73220fdcb4e6f4592295277fbb657d31fd053237c \\\n    --hash=sha256:a83b0e84cc375e3318a808d032510dde99d696a85fe9473fc8575612b63ae951 \\\n    --hash=sha256:c0d25d1d8e531b8343bea0ed811d5d528958f1dcbd37e7245bc686273177ad7e \\\n    --hash=sha256:c29455d2b910b98738131990394da3e50eea8291dfeb4b12de71ecf1fdeb21cb \\\n    --hash=sha256:c432d04376f6d9767a9852ea0def7b47a7bbc8e7af3b16ac9cf9ce02b12851c9 \\\n    --hash=sha256:c88d3299ddeb2b35dcc31753305612db485ab6f1823e37fb29451c8b2732b87e \\\n    --hash=sha256:cb10896a1f7fedaddbccc2017ce6ca9ecaaf990f0973bdfcf405439750118d2c \\\n    --hash=sha256:d033ff0ac3f5400df862a51bdde9bad83561f3739ea0046e68f5401ebfa67c1b \\\n    --hash=sha256:d187566a2cdc726fc80138c3cdb260970fab1c27e99f85452721f7759bbd554d \\\n    --hash=sha256:da5f6f4d7f4940a173e5572791af238cb0b9e21b1aab592bd8b26da4c99f1cd6\n    # via\n    #   -r tools/reef-requirements.in\n    #   accelerate\n    #   chronos-forecasting\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.6 \\\n    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n    # via\n    #   -r tools/reef-requirements.in\n    #   chronos-forecasting\ntriton==3.5.1 \\\n    --hash=sha256:02c770856f5e407d24d28ddc66e33cf026e6f4d360dcb8b2fabe6ea1fc758621 \\\n    --hash=sha256:0b4d2c70127fca6a23e247f9348b8adde979d2e7a20391bfbabaac6aebc7e6a8 \\\n    --hash=sha256:275a045b6ed670dd1bd005c3e6c2d61846c74c66f4512d6f33cc027b11de8fd4 \\\n    --hash=sha256:56765ffe12c554cd560698398b8a268db1f616c120007bfd8829d27139abd24a \\\n    --hash=sha256:5fc53d849f879911ea13f4a877243afc513187bc7ee92d1f2c0f1ba3169e3c94 \\\n    --hash=sha256:61413522a48add32302353fdbaaf92daaaab06f6b5e3229940d21b5207f47579 \\\n    --hash=sha256:8932391d7f93698dfe5bc9bead77c47a24f97329e9f20c10786bb230a9083f56 \\\n    --hash=sha256:bac7f7d959ad0f48c0e97d6643a1cc0fd5786fe61cb1f83b537c6b2d54776478 \\\n    --hash=sha256:d0637b1efb1db599a8e9dc960d53ab6e4637db7d4ab6630a0974705d77b14b60 \\\n    --hash=sha256:d2c6b915a03888ab931a9fd3e55ba36785e1fe70cbea0b40c6ef93b20fc85232 \\\n    --hash=sha256:da47169e30a779bade679ce78df4810fca6d78a955843d2ddb11f226adc517dc \\\n    --hash=sha256:f3f4346b6ebbd4fad18773f5ba839114f4826037c9f2f34e0148894cd5dd3dba \\\n    --hash=sha256:f617aa7925f9ea9968ec2e1adaf93e87864ff51549c8f04ce658f29bbdb71e2d \\\n    --hash=sha256:f63e34dcb32d7bd3a1d0195f60f30d2aee8b08a69a0424189b71017e23dfc3d2\n    # via torch\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   huggingface-hub\n    #   torch\ntzdata==2026.4 \\\n    --hash=sha256:c2169a8b0a7a5e9674da5a135ccdfb2b3e671b333ed9fed17b41f73c34476e81 \\\n    --hash=sha256:f1b8bd365d8d210c55353f4d7f8d6d8561c0ba50d704b700d195a9424bba0d79\n    # via pandas\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via requests\n'

(RUN_ROOT / "requirements.lock").write_text(REQUIREMENTS, encoding="utf-8")
print("Installing the hash-locked environment (a few minutes on a fresh runtime)...", flush=True)
seconds = run_logged([sys.executable, "-m", "uv", "pip", "sync", "--python", PYTHON,
                      "--require-hashes", RUN_ROOT / "requirements.lock"], "setup_sync", echo=False)
print(f"Locked packages installed in {seconds:.0f} s (log: {LOG_DIR / 'setup_sync.log'}).")
print(f"Isolated Python {TARGET_PYTHON} dependencies ready (kernel Python {platform.python_version()});"
      " the notebook kernel does not need a restart.")
_ = (RUN_ROOT / "setup_summary.json").write_text(json.dumps({
    "seconds": time.perf_counter() - SETUP_STARTED,
    "kernel_python": platform.python_version(),
    "environment_python": TARGET_PYTHON,
    "environment_reused": ENV_REUSED,
}))

## 2. Data provenance and integrity

The immutable sample contains all five Philippine regional records from 1985-01-01 through 2026-09-26.
NOAA makes CRW content public domain and asks for attribution. The archive retains exact source bytes and hashes.
It is embedded data, not executable code. There is no fallback to mutable live data on an integrity failure.

Western and Southern each lack 25 dates. Missing dates remain missing; they are never zero stress.
Early 1985 DHW/BAA zeros precede the fields' declared valid starts and are masked.
The legacy BAA categories are retained for provenance, not treated as the newer official alert system.

[NOAA data directory](https://coralreefwatch.noaa.gov/product/vs/data.php) ·
[Rights and citation](https://coralreefwatch.noaa.gov/satellite/docs/recommendations_crw_citation.php)

In [ ]:
# Infrastructure: immutable public NOAA sample, carried inside this notebook.
DATA_MANIFEST = json.loads('{"schema_version": 1, "dataset_id": "noaa-crw-rvs-v3.1-philippines-2026-09-28", "product": "NOAA Coral Reef Watch Regional Virtual Stations", "product_version": "3.1", "retrieval_utc": "2026-09-28T05:26:03.426497+00:00", "coverage_start": "1985-01-01", "coverage_end": "2026-09-26", "archive": {"filename": "noaa_crw_philippines_2026-09-28.zip", "bytes": 1040559, "sha256": "320a04998ff0f0ce4a66264fc83b585c0fafd653844d533468d98a84d6d07b43", "format": "zip"}, "members": [{"filename": "central_philippines.txt", "region_id": "central", "source_url": "https://coralreefwatch.noaa.gov/product/vs/data/central_philippines.txt", "bytes": 1265690, "sha256": "84ce50cf4698d05f8f0e45a524ca381c57b3f8acef24b35bccb5f8e9bc936866", "rows": 15244, "start": "1985-01-01", "end": "2026-09-26", "missing_dates": []}, {"filename": "eastern_philippines.txt", "region_id": "eastern", "source_url": "https://coralreefwatch.noaa.gov/product/vs/data/eastern_philippines.txt", "bytes": 1265690, "sha256": "b4d7c93e8232947f8dfce0c6203f9e019e9e62b8e41d7e8dd2d6d82e35c7ccee", "rows": 15244, "start": "1985-01-01", "end": "2026-09-26", "missing_dates": []}, {"filename": "northern_philippines.txt", "region_id": "northern", "source_url": "https://coralreefwatch.noaa.gov/product/vs/data/northern_philippines.txt", "bytes": 1265691, "sha256": "a963b9a5b13283f8bf8d9b8ce577db887e7ec9792e36b05b8a59c62eb424b3ae", "rows": 15244, "start": "1985-01-01", "end": "2026-09-26", "missing_dates": []}, {"filename": "southern_philippines.txt", "region_id": "southern", "source_url": "https://coralreefwatch.noaa.gov/product/vs/data/southern_philippines.txt", "bytes": 1263616, "sha256": "3b04a206b4c95031c7ad53e6c5a01b1616f05bbf2c4010de9f3155d75dcdd933", "rows": 15219, "start": "1985-01-01", "end": "2026-09-26", "missing_dates": ["2025-01-05", "2025-01-13", "2025-10-02", "2025-10-03", "2025-10-04", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09", "2025-10-10", "2025-10-11", "2025-10-12", "2025-10-13", "2025-10-14", "2025-10-15", "2025-10-16", "2025-10-17", "2025-10-18", "2025-10-19", "2025-10-20", "2025-10-21", "2025-10-22", "2025-10-23", "2026-03-24"]}, {"filename": "western_philippines.txt", "region_id": "western", "source_url": "https://coralreefwatch.noaa.gov/product/vs/data/western_philippines.txt", "bytes": 1263615, "sha256": "5521c5ac1415fdb986b533dd2aa6c0116a736093bbb3b839362956b935daa65f", "rows": 15219, "start": "1985-01-01", "end": "2026-09-26", "missing_dates": ["2025-01-05", "2025-01-13", "2025-10-02", "2025-10-03", "2025-10-04", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09", "2025-10-10", "2025-10-11", "2025-10-12", "2025-10-13", "2025-10-14", "2025-10-15", "2025-10-16", "2025-10-17", "2025-10-18", "2025-10-19", "2025-10-20", "2025-10-21", "2025-10-22", "2025-10-23", "2026-03-24"]}], "original_audit": {"filename": "audit.json", "bytes": 16125, "sha256": "178414d1d835e172bfeb4436acd65a512c326abb67a4a0c32c4cbd11b2d2655b"}, "rights": {"status": "Public domain NOAA Coral Reef Watch website content", "source_url": "https://coralreefwatch.noaa.gov/satellite/docs/recommendations_crw_citation.php", "attribution": "NOAA Coral Reef Watch", "separate_from_code_license": true}, "citation": "NOAA Coral Reef Watch. 2019. NOAA Coral Reef Watch Version 3.1 Daily Global 5km Satellite Coral Bleaching Heat Stress Monitoring Products: Regional Virtual Stations, Northern, Central, Western, Eastern and Southern Philippines; 1 January 1985 to 26 September 2026. NOAA Coral Reef Watch, College Park, Maryland, USA. Accessed 28 September 2026. https://coralreefwatch.noaa.gov/product/vs/data.php", "scientific_contract": {"interpretation": "Daily moving-pixel regional 90th-percentile HotSpot summary; not a fixed sensor, regional-mean SST, or observed bleaching label", "dhw_first_valid_date": "1985-03-25", "baa_first_valid_date": "1985-03-31", "dhw_formula": "sum(HotSpot if HotSpot >= 1 C else 0) over 84 calendar days / 7", "missing_policy": "Reindex daily; preserve gaps; never zero-fill missing observations", "alert_scheme": "Legacy BAA; not interchangeable with current global alert categories"}, "distribution": {"default": "Notebook embeds this exact ZIP as base64 data and verifies archive/member hashes before parsing", "spec_amendment": "An embedded immutable data-only archive replaces the proposed remote immutable asset download, avoiding an unpublished URL. No executable source is fetched; no live-data fallback.", "live_refresh": "Different dataset version requiring a new audit and manifest; not a default fallback"}}')
MODEL_MANIFEST = json.loads('{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "chronos-2", "modelId": "amazon/chronos-2", "revision": "95a9710e2596287d08352589f42634fa5abdf0a7", "files": [{"path": "config.json", "bytes": 1067, "sha256": "ef1143bfdc9c0376d9a056eefca46cb4b1ec3d0ffacd541ff56feb40fb708031"}, {"path": "model.safetensors", "bytes": 477930472, "sha256": "ddcda3c7508bf2528087723e98a20707cc04b7f370ae275a9fd88078ddba4f42"}], "totalBytes": 477931539}')
ARCHIVE_B64 = '...'
payload = base64.b64decode(ARCHIVE_B64, validate=True)
if hashlib.sha256(payload).hexdigest() != DATA_MANIFEST["archive"]["sha256"]:
    raise ValueError("Embedded sample integrity failure")
(RUN_ROOT / DATA_MANIFEST["archive"]["filename"]).write_bytes(payload)
(RUN_ROOT / "dataset_manifest.json").write_text(json.dumps(DATA_MANIFEST, indent=2))
(RUN_ROOT / "model_manifest.json").write_text(json.dumps(MODEL_MANIFEST, indent=2))
print("NOAA Coral Reef Watch", DATA_MANIFEST["coverage_start"], "through", DATA_MANIFEST["coverage_end"])

## 3. Infrastructure: visible implementation

The following three cells carry the validation/mathematics, pinned model adapter and stage runner as readable
Python source. They are written locally inside this runtime; no DIMER repository source is downloaded or imported.
The core stage helpers expose dates, baselines, DHW accumulation and metrics. Each stage runs in a separate process,
so no hidden kernel model state is required. Chronos is frozen; no fine-tuning is performed.

In [ ]:
# Infrastructure: reef_core.py
CORE_SOURCE = r'''
"""Standalone numerical contract for the Philippine reef capstone (CPU only)."""

import io
import json

import numpy as np
import pandas as pd

REEF_COLUMNS = [
    "YYYY",
    "MM",
    "DD",
    "SST_MIN",
    "SST_MAX",
    "SST@90th_HS",
    "SSTA@90th_HS",
    "90th_HS>0",
    "DHW_from_90th_HS>1",
    "BAA_7day_max",
]
REEF_NAMES = [
    "sst_min_c",
    "sst_max_c",
    "representative_sst_c",
    "representative_ssta_c",
    "hotspot_c",
    "dhw_c_weeks",
    "legacy_baa_7day",
]


FORECAST_KEYS = ["region_id", "origin", "arm", "lead"]
QUANTILE_COLUMNS = [
    "model_q10",
    "model_q50",
    "model_q90",
    "quantile_crossing_c",
    "raw_q10",
    "raw_q50",
    "raw_q90",
    "q10",
    "q50",
    "q90",
]
QUANTILE_ARMS = {"chronos", "chronos_180"}


def json_native(value: object) -> object:
    """JSON default for scientific records: native numbers, never blanket strings."""
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, pd.Timestamp):
        return str(value.date()) if value == value.normalize() else value.isoformat()
    raise TypeError(f"Not JSON-serializable: {type(value).__name__}")


def _finite_nonnegative(values: object, label: str) -> np.ndarray:
    try:
        arr = np.asarray(values, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: numeric values required") from exc
    if not np.isfinite(arr).all() or (arr < 0).any():
        raise ValueError(f"{label}: finite nonnegative values required")
    return arr


def parse_noaa(text: str, region: str) -> pd.DataFrame:
    """Parse exact NOAA RVS schema and its unusual year/day/month metadata."""
    lines = text.splitlines()
    headers = [i for i, line in enumerate(lines) if line.split() == REEF_COLUMNS]
    if len(headers) != 1:
        raise ValueError("Missing or malformed NOAA column header")
    meta = {}
    for label in ["Name", "First Valid DHW Date", "First Valid BAA Date"]:
        matches = [i for i, line in enumerate(lines[: headers[0]]) if line.strip() == label + ":"]
        if len(matches) != 1 or matches[0] + 1 >= headers[0]:
            raise ValueError(f"Missing metadata: {label}")
        meta[label] = lines[matches[0] + 1].strip()
    try:
        valid_dhw = pd.to_datetime(meta["First Valid DHW Date"], format="%Y %d %m")
        valid_baa = pd.to_datetime(meta["First Valid BAA Date"], format="%Y %d %m")
    except ValueError as exc:
        raise ValueError("Malformed validity date; expected year day month") from exc
    body = [line for line in lines[headers[0] + 1 :] if line.strip()]
    if not body or any(len(line.split()) != 10 for line in body):
        raise ValueError("NOAA data rows must contain exactly ten fields")
    raw = pd.read_csv(io.StringIO("\n".join(body)), sep=r"\s+", names=REEF_COLUMNS)
    raw = raw.apply(pd.to_numeric, errors="raise")
    if not np.isfinite(raw.to_numpy(dtype=float)).all():
        raise ValueError("NOAA data contains nonfinite values")
    parts = raw[["YYYY", "MM", "DD"]]
    if not (parts.to_numpy() == parts.to_numpy().astype(int)).all():
        raise ValueError("Date components must be integers")
    dates = pd.to_datetime(
        parts.rename(columns={"YYYY": "year", "MM": "month", "DD": "day"}), errors="raise"
    )
    if dates.duplicated().any():
        raise ValueError(f"Duplicate dates: {dates[dates.duplicated()].astype(str).tolist()[:5]}")
    out = raw.iloc[:, 3:].copy()
    out.columns = REEF_NAMES
    _finite_nonnegative(out[["hotspot_c", "dhw_c_weeks"]], "NOAA HotSpot/DHW")
    if not out["legacy_baa_7day"].isin(range(5)).all():
        raise ValueError("Legacy BAA must be an integer from zero through four")
    if (
        (out.sst_min_c > out.representative_sst_c) | (out.sst_max_c < out.representative_sst_c)
    ).any():
        raise ValueError("Representative SST outside regional bounds")
    out.insert(0, "date", dates)
    out.insert(0, "region_id", region)
    out["raw_dhw_c_weeks"] = out.dhw_c_weeks
    out["raw_legacy_baa_7day"] = out.legacy_baa_7day
    out["dhw_valid"] = out.date >= valid_dhw
    out["baa_valid"] = out.date >= valid_baa
    out.loc[~out.dhw_valid, "dhw_c_weeks"] = np.nan
    out.loc[~out.baa_valid, "legacy_baa_7day"] = np.nan
    out.attrs["metadata"] = meta
    return out.sort_values("date").reset_index(drop=True)


def read_byod_csv(path: object) -> pd.DataFrame:
    """Read BYOD text literally: region IDs such as 001 or NA keep their exact spelling."""
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, na_filter=False)
    for column in ["hotspot_c", "dhw_c_weeks"]:
        if column in frame:
            if (frame[column].str.strip() == "").any():
                raise ValueError(f"{column}: blank values are not allowed")
            frame[column] = _finite_nonnegative(frame[column], column)
    return frame


def validate_byod(
    frame: pd.DataFrame, units: str = "degC", semantics_confirmed: bool = False
) -> pd.DataFrame:
    """Refuse SST substitutions, timestamps, duplicate/gapped or short contexts."""
    required = {"region_id", "date", "hotspot_c"}
    if not required.issubset(frame.columns) or set(frame.columns) - required - {"dhw_c_weeks"}:
        raise ValueError("Expected region_id,date,hotspot_c and optional dhw_c_weeks only")
    if units != "degC" or not semantics_confirmed:
        raise ValueError("Confirm NOAA threshold-relative HotSpot semantics and degC units")
    out = frame.copy()
    if (
        out.empty
        or out.region_id.isna().any()
        or (out.region_id.astype(str).str.strip() == "").any()
    ):
        raise ValueError("Nonempty region IDs and data required")
    if not out.date.astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").all():
        raise ValueError("Dates must be exactly YYYY-MM-DD without a time or timezone")
    out["date"] = pd.to_datetime(out.date, format="%Y-%m-%d", errors="raise")
    if out.duplicated(["region_id", "date"]).any():
        raise ValueError("Duplicate region/date rows")
    for col in ["hotspot_c"] + (["dhw_c_weeks"] if "dhw_c_weeks" in out else []):
        out[col] = _finite_nonnegative(out[col], col)
    for region, group in out.groupby("region_id"):
        group = group.sort_values("date")
        if len(group) < 365 or not group.date.diff().iloc[1:].eq(pd.Timedelta(days=1)).all():
            raise ValueError(f"{region}: at least 365 continuous daily observations required")
        if "dhw_c_weeks" in group:
            reconstructed = reconstruct_dhw(group.set_index("date").hotspot_c)
            if not np.allclose(
                reconstructed.dropna(),
                group.set_index("date").loc[reconstructed.notna(), "dhw_c_weeks"],
                atol=1e-4,
                rtol=0,
            ):
                raise ValueError(
                    f"{region}: DHW disagrees with calendar-aware HotSpot accumulation"
                )
    return out.sort_values(["region_id", "date"]).reset_index(drop=True)


def daily_panel(frames: list[pd.DataFrame]) -> pd.DataFrame:
    """Preserve missing calendar days explicitly; never fill numerical outcomes."""
    data = pd.concat(frames, ignore_index=True)
    if data.duplicated(["region_id", "date"]).any():
        raise ValueError("Duplicate region/date in daily panel")
    groups = []
    for region, group in data.groupby("region_id", sort=True):
        group = group.set_index("date").drop(columns="region_id").sort_index()
        group["observed"] = True
        group = group.reindex(
            pd.date_range(group.index.min(), group.index.max(), freq="D", name="date")
        )
        group["observed"] = group.observed.eq(True)
        group["missing_day"] = ~group.observed
        group["region_id"] = region
        groups.append(group.reset_index())
    return pd.concat(groups, ignore_index=True).set_index(["region_id", "date"]).sort_index()


def reconstruct_dhw(hotspot: pd.Series) -> pd.Series:
    """Inclusive 84-calendar-day sum with >=1 degree threshold; gaps remain null."""
    if not isinstance(hotspot.index, pd.DatetimeIndex) or hotspot.index.has_duplicates:
        raise ValueError("Unique DatetimeIndex required")
    hs = hotspot.sort_index().reindex(
        pd.date_range(hotspot.index.min(), hotspot.index.max(), freq="D")
    )
    if (hs.dropna() < 0).any() or np.isinf(hs.to_numpy()).any():
        raise ValueError("Invalid HotSpot")
    heat = hs.where(hs >= 1, 0).where(hs.notna())
    return heat.rolling(84, min_periods=84).sum() / 7


def context_at(panel: pd.DataFrame, region: str, origin: object, length: int = 365) -> pd.Series:
    """Only observations up to origin, with calendar continuity required."""
    origin = pd.Timestamp(origin)
    dates = pd.date_range(origin - pd.Timedelta(days=length - 1), origin)
    context = panel.xs(region).hotspot_c.reindex(dates)
    _finite_nonnegative(context, "context")
    return context


def origin_manifest(panel: pd.DataFrame) -> pd.DataFrame:
    """All planned 1st/15th origins, including split-boundary and gap refusals."""
    rows = []
    for region in panel.index.get_level_values("region_id").unique():
        data = panel.xs(region)
        for year in [2023, 2024, 2025]:
            for month in range(1, 13):
                for day in [1, 15]:
                    origin = pd.Timestamp(year, month, day)
                    targets = pd.date_range(origin + pd.Timedelta(days=1), periods=28)
                    context_dates = pd.date_range(origin - pd.Timedelta(days=364), origin)
                    reasons = []
                    # Each annual test report uses complete horizons within that year.
                    if targets[-1].year != year:
                        reasons.append("outcome_crosses_year_boundary")
                    if data.hotspot_c.reindex(context_dates).isna().any():
                        reasons.append("incomplete_365_day_context")
                    if data.hotspot_c.reindex(targets).isna().any():
                        reasons.append("incomplete_28_day_outcome")
                    if "dhw_c_weeks" not in data or data.dhw_c_weeks.reindex(targets).isna().any():
                        reasons.append("invalid_dhw_outcome")
                    rows.append(
                        dict(
                            region_id=region,
                            origin=origin,
                            year=year,
                            split="validation" if year == 2023 else "test",
                            eligible=not reasons,
                            reason=";".join(reasons),
                            context_start=context_dates[0],
                            target_end=targets[-1],
                        )
                    )
    return pd.DataFrame(rows)


def fit_seasonal(panel: pd.DataFrame) -> pd.DataFrame:
    """Fit only 2017-2022, using month/day rather than shifting leap-year DOY."""
    frame = panel.reset_index()
    frame = frame[frame.date.between("2017-01-01", "2022-12-31")].copy()
    frame["month"] = frame.date.dt.month
    frame["day"] = frame.date.dt.day
    result = frame.groupby(["region_id", "month", "day"], as_index=False).hotspot_c.mean()
    result = result[~((result.month == 2) & (result.day == 29))]
    for region in frame.region_id.unique():
        vals = result[
            (result.region_id == region)
            & (
                ((result.month == 2) & (result.day == 28))
                | ((result.month == 3) & (result.day == 1))
            )
        ].hotspot_c
        if len(vals) != 2 or not np.isfinite(vals).all():
            raise ValueError("Seasonal reference lacks February/March boundary")
        result = pd.concat(
            [
                result,
                pd.DataFrame(
                    [dict(region_id=region, month=2, day=29, hotspot_c=float(vals.mean()))]
                ),
            ],
            ignore_index=True,
        )
    if result.hotspot_c.isna().any() or any(len(g) != 366 for _, g in result.groupby("region_id")):
        raise ValueError("Incomplete seasonal development reference")
    return result.sort_values(["region_id", "month", "day"]).reset_index(drop=True)


def baseline_forecasts(
    context: pd.Series, dates: object, seasonal: pd.DataFrame, region: str
) -> dict:
    hs = _finite_nonnegative(context, "context")
    dates = pd.DatetimeIndex(dates)
    lookup = seasonal[seasonal.region_id == region].set_index(["month", "day"]).hotspot_c
    return {
        "persistence": np.repeat(hs[-1], len(dates)),
        "seasonal": np.array([lookup.loc[(d.month, d.day)] for d in dates], dtype=float),
    }


def compose_dhw(context: object, predictions: object) -> dict:
    """Separate known history from forecast heat, without future observations."""
    history = _finite_nonnegative(context, "context")
    pred = _finite_nonnegative(predictions, "predictions")
    if history.ndim != 1 or pred.ndim != 1 or len(history) < 83:
        raise ValueError("DHW requires a 1D context of at least 83 consecutive days")
    hist_heat = np.where(history >= 1, history, 0)
    pred_heat = np.where(pred >= 1, pred, 0)
    known, new = [], []
    for i in range(len(pred)):
        remaining = max(83 - i, 0)
        known.append(float(hist_heat[-remaining:].sum() / 7) if remaining else 0.0)
        new.append(float(pred_heat[max(0, i - 83) : i + 1].sum() / 7))
    return {
        "dhw": np.array(known) + np.array(new),
        "known": np.array(known),
        "predicted": np.array(new),
    }


def rearrange_quantiles(quantiles: object) -> tuple[np.ndarray, np.ndarray]:
    """Monotone rearrangement: sort each day's q10/q50/q90 and report the largest crossing.

    Chronos-2 predicts its quantiles independently and does not enforce their order, so a
    crossing is a model property, not corrupt output. Sorting never increases pinball loss
    (Chernozhukov, Fernandez-Val and Galichon, 2010); the crossing size is kept for audit.
    """
    q = np.asarray(quantiles, dtype=float)
    if q.ndim != 2 or q.shape[1] != 3 or not np.isfinite(q).all():
        raise ValueError("Quantiles must be finite (H,3) q10,q50,q90")
    pairs = [q[:, 0] - q[:, 1], q[:, 1] - q[:, 2], q[:, 0] - q[:, 2]]
    crossing = np.maximum.reduce([np.zeros(len(q))] + pairs)
    return np.sort(q, axis=1), crossing


def forecast_frame(
    panel: pd.DataFrame,
    region: str,
    origin: object,
    arm: str,
    raw: object,
    split: str,
    quantiles: object = None,
) -> pd.DataFrame:
    """Build 28 daily predictions; observed targets are attached for scoring only."""
    origin = pd.Timestamp(origin)
    values = np.asarray(raw, dtype=float)
    if values.shape != (28,) or not np.isfinite(values).all():
        raise ValueError("Each arm must return exactly 28 finite predictions")
    if quantiles is not None:
        model_q = np.asarray(quantiles, dtype=float)
        if model_q.shape != (28, 3) or not np.isfinite(model_q).all():
            raise ValueError("Quantiles must be finite (28,3) q10,q50,q90")
        if not np.allclose(model_q[:, 1], values, atol=1e-6, rtol=0):
            raise ValueError("Point forecast must equal the model median")
        q, crossing = rearrange_quantiles(model_q)
        # The point forecast is the median of the rearranged (ordered) quantiles.
        values = q[:, 1]
    constrained = np.maximum(values, 0)
    dates = pd.date_range(origin + pd.Timedelta(days=1), periods=28)
    data = panel.xs(region)
    dhw = compose_dhw(context_at(panel, region, origin), constrained)
    out = pd.DataFrame(
        dict(
            region_id=region,
            origin=origin,
            target_date=dates,
            lead=np.arange(1, 29),
            arm=arm,
            split=split,
            raw_hotspot_c=values,
            hotspot_c=constrained,
            dhw_c_weeks=dhw["dhw"],
            dhw_known=dhw["known"],
            dhw_predicted=dhw["predicted"],
            actual_hotspot_c=data.hotspot_c.reindex(dates).to_numpy(),
            actual_dhw_c_weeks=data.dhw_c_weeks.reindex(dates).to_numpy(),
            origin_dhw_c_weeks=float(data.dhw_c_weeks.loc[origin]),
        )
    )
    if quantiles is not None:
        for i, label in enumerate(["q10", "q50", "q90"]):
            out["model_" + label] = model_q[:, i]
        out["quantile_crossing_c"] = crossing
        for i, label in enumerate(["q10", "q50", "q90"]):
            out["raw_" + label] = q[:, i]
            out[label] = np.maximum(q[:, i], 0)
    return out


def validate_forecast_grid(
    frame: pd.DataFrame, plan: pd.DataFrame, arms: object, panel: pd.DataFrame | None = None
) -> pd.DataFrame:
    """Require exactly one row per planned region x origin x arm x lead, with date identity.

    ``plan`` holds the frozen eligible origins (region_id, origin, split). Missing whole arms,
    regions or origins, extra keys, shifted target dates, inconsistent source-derived
    references and absent Chronos quantiles are refused before any score is calculated.
    """
    arms = list(arms)
    needed = FORECAST_KEYS + ["split", "target_date", "hotspot_c", "raw_hotspot_c"]
    needed += ["actual_hotspot_c", "actual_dhw_c_weeks", "origin_dhw_c_weeks", "dhw_c_weeks"]
    missing_columns = [c for c in needed if c not in frame]
    if QUANTILE_ARMS & set(arms):
        missing_columns += [c for c in QUANTILE_COLUMNS if c not in frame]
    if missing_columns:
        raise ValueError(f"Forecast file lacks required columns: {missing_columns}")
    data = frame.copy()
    data["origin"] = pd.to_datetime(data.origin)
    data["target_date"] = pd.to_datetime(data.target_date)
    data["lead"] = pd.to_numeric(data.lead, errors="raise")
    if data.duplicated(FORECAST_KEYS).any():
        raise ValueError("Duplicate forecast rows")
    planned = plan[["region_id", "origin", "split"]].copy()
    planned["origin"] = pd.to_datetime(planned.origin)
    if planned.duplicated(["region_id", "origin"]).any():
        raise ValueError("Duplicate planned origins")
    expected = planned.merge(pd.DataFrame({"arm": arms}), how="cross").merge(
        pd.DataFrame({"lead": np.arange(1, 29)}), how="cross"
    )
    key = FORECAST_KEYS + ["split"]
    joined = expected.merge(data[key], how="outer", on=key, indicator=True)
    absent = joined[joined._merge == "left_only"]
    extra = joined[joined._merge == "right_only"]
    if len(absent):
        groups = absent.drop_duplicates(["region_id", "origin", "arm"])
        raise ValueError(
            f"Missing planned forecasts: {len(absent)} rows in {len(groups)} region-origin-arm "
            f"groups, e.g. {groups.iloc[0][['region_id', 'arm']].tolist()} "
            f"{groups.iloc[0].origin.date()}"
        )
    if len(extra):
        raise ValueError(f"Unplanned forecast rows (key, arm or split): {len(extra)}")
    shifted = data.target_date != data.origin + pd.to_timedelta(data.lead, unit="D")
    if shifted.any():
        raise ValueError(f"target_date must equal origin + lead; {int(shifted.sum())} rows differ")
    model_rows = data[data.arm.isin(QUANTILE_ARMS)]
    if len(model_rows):
        values = model_rows[QUANTILE_COLUMNS].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError("Required Chronos quantiles are missing or nonfinite")
    if panel is not None:
        idx = pd.MultiIndex.from_arrays([data.region_id, data.target_date])
        origin_idx = pd.MultiIndex.from_arrays([data.region_id, data.origin])
        references = {
            "actual_hotspot_c": panel.hotspot_c.reindex(idx).to_numpy(),
            "actual_dhw_c_weeks": panel.dhw_c_weeks.reindex(idx).to_numpy(),
            "origin_dhw_c_weeks": panel.dhw_c_weeks.reindex(origin_idx).to_numpy(),
        }
        for column, truth in references.items():
            if not np.allclose(data[column].to_numpy(dtype=float), truth, atol=1e-9, rtol=0):
                raise ValueError(f"{column} disagrees with the frozen source data")
    return data


def evaluate_forecasts(frame: pd.DataFrame, expected_arms: object = None) -> dict:
    """Descriptive region/macro metrics; no iid uncertainty or ecological labels."""
    metrics, thresholds, paired = [], [], []
    # Illustrative outlook rows are deliberately not evaluated.
    evaluated = frame[frame.split.isin(["validation", "test"])].copy()
    if evaluated.empty:
        raise ValueError("No validation/test forecasts to evaluate")
    if expected_arms is not None and set(evaluated.arm) != set(expected_arms):
        raise ValueError("Missing or unexpected model arms")
    if evaluated.duplicated(["region_id", "origin", "arm", "lead"]).any():
        raise ValueError("Duplicate forecast rows")
    for _, group in evaluated.groupby(["split", "region_id", "origin", "arm"]):
        if sorted(group.lead.tolist()) != list(range(1, 29)):
            raise ValueError("Missing model outputs or incomplete horizon")
        if (
            not np.isfinite(
                group[["hotspot_c", "dhw_c_weeks", "actual_hotspot_c", "actual_dhw_c_weeks"]]
            )
            .all()
            .all()
        ):
            raise ValueError("Missing/nonfinite model outputs or outcomes")
        if (
            (group[["hotspot_c", "dhw_c_weeks", "actual_hotspot_c", "actual_dhw_c_weeks"]] < 0)
            .any()
            .any()
        ):
            raise ValueError("Negative constrained forecasts or outcomes")
        qcols = ["q10", "q50", "q90"]
        if all(c in group for c in qcols) and group[qcols].notna().any().any():
            qvalues = group[qcols].to_numpy(dtype=float)
            if not np.isfinite(qvalues).all() or (np.diff(qvalues, axis=1) < 0).any():
                raise ValueError("Invalid serialized quantiles")
    for split, sg in evaluated.groupby("split"):
        # Compare whole region-origin sets, so an arm absent from a region cannot hide.
        arm_keys = [set(zip(g.region_id, g.origin, strict=True)) for _, g in sg.groupby("arm")]
        if any(s != arm_keys[0] for s in arm_keys):
            raise ValueError("Models must use identical comparison origins")
        panels = [("full", sg)] + (
            [(str(y), sg[pd.to_datetime(sg.origin).dt.year == y]) for y in [2024, 2025]]
            if split == "test"
            else []
        )
        for period, pg in panels:
            for (region, arm), group in pg.groupby(["region_id", "arm"]):
                for horizon in [7, 14, 28]:
                    prefix = group[group.lead <= horizon]
                    endpoint = group[group.lead == horizon]
                    err = prefix.hotspot_c - prefix.actual_hotspot_c
                    common = dict(
                        split=split,
                        period=period,
                        region_id=region,
                        arm=arm,
                        horizon=int(horizon),
                        origins=int(prefix.origin.nunique()),
                        days=int(len(prefix)),
                    )
                    row = dict(
                        common,
                        hotspot_mae=float(abs(err).mean()),
                        hotspot_rmse=float(np.sqrt((err**2).mean())),
                        dhw_endpoint_mae=float(
                            abs(endpoint.dhw_c_weeks - endpoint.actual_dhw_c_weeks).mean()
                        ),
                    )
                    stress = prefix[prefix.actual_hotspot_c >= 1]
                    row["high_stress_days"] = len(stress)
                    row["high_stress_mae"] = (
                        float(abs(stress.hotspot_c - stress.actual_hotspot_c).mean())
                        if len(stress)
                        else None
                    )
                    if all(c in prefix and prefix[c].notna().all() for c in ["q10", "q50", "q90"]):
                        for level, col in [(0.1, "q10"), (0.5, "q50"), (0.9, "q90")]:
                            residual = prefix.actual_hotspot_c - prefix[col]
                            row[f"pinball_{col}"] = float(
                                np.maximum(level * residual, (level - 1) * residual).mean()
                            )
                        row["coverage_10_90"] = float(
                            (
                                (prefix.actual_hotspot_c >= prefix.q10)
                                & (prefix.actual_hotspot_c <= prefix.q90)
                            ).mean()
                        )
                        row["width_10_90"] = float((prefix.q90 - prefix.q10).mean())
                    metrics.append(row)
                    for threshold in [4.0, 8.0]:
                        for subset in ["all", "new_exceedance"]:
                            eg = (
                                endpoint
                                if subset == "all"
                                else endpoint[endpoint.origin_dhw_c_weeks < threshold]
                            )
                            truth, pred = (
                                eg.actual_dhw_c_weeks >= threshold,
                                eg.dhw_c_weeks >= threshold,
                            )
                            tp, fp, fn, tn = [
                                int(mask.sum())
                                for mask in [
                                    truth & pred,
                                    ~truth & pred,
                                    truth & ~pred,
                                    ~truth & ~pred,
                                ]
                            ]
                            thresholds.append(
                                dict(
                                    common,
                                    threshold=threshold,
                                    subset=subset,
                                    n=len(eg),
                                    tp=tp,
                                    fp=fp,
                                    fn=fn,
                                    tn=tn,
                                    positive_support=tp + fn,
                                    precision=tp / (tp + fp) if tp + fp else None,
                                    recall=tp / (tp + fn) if tp + fn else None,
                                    precision_reason=None if tp + fp else "no_predicted_positives",
                                    recall_reason=None if tp + fn else "no_observed_positives",
                                )
                            )
                    for origin, og in prefix.groupby("origin"):
                        paired.append(
                            dict(
                                common,
                                origin=str(pd.Timestamp(origin).date()),
                                hotspot_mae=float(abs(og.hotspot_c - og.actual_hotspot_c).mean()),
                            )
                        )
            # DHW persistence has no HotSpot score.
            unique = pg.drop_duplicates(["region_id", "origin", "lead"])
            for region, group in unique.groupby("region_id"):
                for horizon in [7, 14, 28]:
                    endpoint = group[group.lead == horizon]
                    metrics.append(
                        dict(
                            split=split,
                            period=period,
                            region_id=region,
                            arm="dhw_persistence",
                            horizon=horizon,
                            origins=int(endpoint.origin.nunique()),
                            days=int(len(endpoint)),
                            dhw_endpoint_mae=float(
                                abs(
                                    endpoint.origin_dhw_c_weeks - endpoint.actual_dhw_c_weeks
                                ).mean()
                            ),
                        )
                    )
                    for threshold in [4.0, 8.0]:
                        for subset in ["all", "new_exceedance"]:
                            eg = (
                                endpoint
                                if subset == "all"
                                else endpoint[endpoint.origin_dhw_c_weeks < threshold]
                            )
                            truth, pred = (
                                eg.actual_dhw_c_weeks >= threshold,
                                eg.origin_dhw_c_weeks >= threshold,
                            )
                            tp, fp, fn, tn = [
                                int(mask.sum())
                                for mask in [
                                    truth & pred,
                                    ~truth & pred,
                                    truth & ~pred,
                                    ~truth & ~pred,
                                ]
                            ]
                            thresholds.append(
                                dict(
                                    split=split,
                                    period=period,
                                    region_id=region,
                                    arm="dhw_persistence",
                                    horizon=horizon,
                                    origins=int(endpoint.origin.nunique()),
                                    days=len(endpoint),
                                    threshold=threshold,
                                    subset=subset,
                                    n=len(eg),
                                    tp=tp,
                                    fp=fp,
                                    fn=fn,
                                    tn=tn,
                                    positive_support=tp + fn,
                                    precision=tp / (tp + fp) if tp + fp else None,
                                    recall=tp / (tp + fn) if tp + fn else None,
                                    precision_reason=None if tp + fp else "no_predicted_positives",
                                    recall_reason=None if tp + fn else "no_observed_positives",
                                )
                            )
    table = pd.DataFrame(metrics)
    numeric = [
        c
        for c in table
        if c.endswith("mae")
        or c.endswith("rmse")
        or c.startswith("pinball_")
        or c in ["coverage_10_90", "width_10_90"]
    ]
    macro = table.groupby(["split", "period", "arm", "horizon"], as_index=False)[numeric].mean()
    support = (
        table.groupby(["split", "period", "arm", "horizon"])
        .agg(regions=("region_id", "nunique"), origins=("origins", "sum"), days=("days", "sum"))
        .reset_index()
    )
    macro = macro.merge(support)
    error_table = pd.DataFrame(paired)
    paired_differences = []
    for keys, group in error_table.groupby(["split", "period", "region_id", "horizon", "origin"]):
        errors = group.set_index("arm").hotspot_mae
        if "chronos" in errors or "chronos2" in errors or "chronos-2" in errors:
            model_arm = next(a for a in ["chronos", "chronos2", "chronos-2"] if a in errors)
            for baseline in ["persistence", "seasonal"]:
                if baseline in errors:
                    paired_differences.append(
                        dict(
                            zip(
                                ["split", "period", "region_id", "horizon", "origin"],
                                keys,
                                strict=True,
                            ),
                            arm=model_arm,
                            baseline=baseline,
                            mae_difference=float(errors[model_arm] - errors[baseline]),
                        )
                    )
    report = dict(
        metrics=json.loads(table.to_json(orient="records")),
        macro=json.loads(macro.to_json(orient="records")),
        thresholds=thresholds,
        per_origin_errors=paired,
        paired_differences=paired_differences,
    )
    # Unavailable metrics are null, never NaN; NumPy scalars become native JSON numbers so a
    # serialized report equals its fresh recomputation under any pandas version.
    return json.loads(json.dumps(report, allow_nan=False, default=json_native))
'''

In [ ]:
# Infrastructure: reef_models.py
MODEL_SOURCE = r'''
"""Pinned upstream Chronos adapter, carried visibly in the standalone notebook."""


import hashlib
import os
import random
import urllib.request
from pathlib import Path

import numpy as np


def reef_sha(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def reef_snapshot(manifest: dict, cache: Path) -> Path:
    """Download only declared immutable weight/config files and verify cache hits."""
    import re

    if not re.fullmatch(r"[0-9a-f]{40}", manifest["revision"]):
        raise ValueError("An immutable model revision is required")
    if manifest["modelId"] != "amazon/chronos-2":
        raise ValueError("Unexpected model identity")
    root = cache / manifest["revision"]
    root.mkdir(parents=True, exist_ok=True)
    for item in manifest["files"]:
        if item["path"] not in {"config.json", "model.safetensors"}:
            raise ValueError("Only configuration and safetensors are allowed")
        path = root / item["path"]
        if path.is_symlink() or root.is_symlink():
            raise ValueError("Model cache must not use symlinks")
        if not path.exists():
            url = (
                f"https://huggingface.co/{manifest['modelId']}/resolve/"
                f"{manifest['revision']}/{item['path']}"
            )
            temporary = path.with_suffix(".partial")
            total = 0
            with urllib.request.urlopen(url, timeout=120) as response, temporary.open("wb") as out:
                while chunk := response.read(1 << 20):
                    total += len(chunk)
                    if total > item["bytes"]:
                        raise ValueError("Model download exceeds pinned size")
                    out.write(chunk)
            if total != item["bytes"] or reef_sha(temporary) != item["sha256"]:
                raise ValueError("Model download integrity failure")
            temporary.replace(path)
        if path.stat().st_size != item["bytes"] or reef_sha(path) != item["sha256"]:
            raise ValueError(f"Model integrity failure: {item['path']}")
    return root


def reef_load_model(manifest: dict, cache: Path):
    """Hosted GPU execution only; never silently substitute a different model."""
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    from importlib.metadata import version

    import torch
    from chronos import Chronos2Pipeline

    if version("chronos-forecasting") != "2.3.1":
        raise RuntimeError("Expected chronos-forecasting 2.3.1")
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab T4 GPU for model stages")
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.use_deterministic_algorithms(True)
    root = reef_snapshot(manifest, cache)
    return Chronos2Pipeline.from_pretrained(
        str(root),
        device_map="cuda",
        dtype=torch.float32,
        local_files_only=True,
    )


def reef_predict(model, history, horizon: int = 28) -> np.ndarray:
    """Return H by 3 raw q10/q50/q90 as the model emits them; one series per call."""
    values = np.asarray(history, dtype=np.float32)
    if values.ndim != 1 or not np.isfinite(values).all() or len(values) < 84:
        raise ValueError("Expected a finite one-dimensional daily history")
    if not 1 <= horizon <= 30:
        raise ValueError("Supported horizon is 1–30 days")
    quantiles, _ = model.predict_quantiles(
        [values.copy()],
        prediction_length=horizon,
        quantile_levels=[0.1, 0.5, 0.9],
        batch_size=1,
        context_length=len(values),
    )
    if len(quantiles) != 1:
        raise ValueError("Unexpected series count")
    result = quantiles[0].detach().float().cpu().numpy()
    if result.shape != (1, horizon, 3) or not np.isfinite(result).all():
        raise ValueError("Invalid quantile dimensions or nonfinite prediction")
    # Quantiles are returned in model order; crossings are rearranged and recorded downstream.
    return result[0].astype(float)
'''

In [ ]:
# Infrastructure: reef_runtime.py
RUNTIME_SOURCE = r'''
# ruff: noqa: F821
"""Standalone stages; core/model names are linked by the notebook builder and tested together."""


import argparse
import hashlib
import io
import json
import platform
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd


def reef_json(path: Path, value: object) -> None:
    path.write_text(
        json.dumps(value, indent=2, allow_nan=False, default=json_native) + "\n", encoding="utf-8"
    )


def reef_read(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def reef_load_data(root: Path):
    manifest = reef_read(root / "dataset_manifest.json")
    raw = (root / manifest["archive"]["filename"]).read_bytes()
    if (
        len(raw) != manifest["archive"]["bytes"]
        or hashlib.sha256(raw).hexdigest() != manifest["archive"]["sha256"]
    ):
        raise ValueError("Frozen NOAA archive integrity failure")
    frames = []
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        expected = {m["filename"] for m in manifest["members"]}
        if len(archive.namelist()) != len(expected) or set(archive.namelist()) != expected:
            raise ValueError("Unexpected or duplicate data members")
        for member in manifest["members"]:
            info = archive.getinfo(member["filename"])
            if info.file_size != member["bytes"]:
                raise ValueError("Unexpected uncompressed size")
            data = archive.read(member["filename"])
            if hashlib.sha256(data).hexdigest() != member["sha256"]:
                raise ValueError("NOAA member integrity failure")
            frames.append(parse_noaa(data.decode("utf-8"), member["region_id"]))
    return daily_panel(frames)


def reef_records(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    for name in ("origin", "target_date", "date"):
        if name in frame:
            frame[name] = pd.to_datetime(frame[name])
    return frame


# Stage order and outputs. Restarting a stage removes its own and every later stage's
# records, so a completion file can never describe outputs from an earlier attempt.
REEF_STAGES = {
    "prepare": [
        "origin_manifest.csv",
        "seasonal_reference.csv",
        "experiment_config.json",
        "dataset_audit.json",
        "byod_example.csv",
    ],
    "baselines": ["baselines_forecasts.csv", "baselines_summary.json"],
    "validation": ["validation_forecasts.csv", "validation_summary.json", "validation_parity.json"],
    "activity": ["activity_forecasts.csv", "activity_summary.json"],
    "compare": ["validation_tables.txt"],
    "lock": ["budget_check.json", "experiment_lock.json"],
    "test": ["test_forecasts.csv", "test_summary.json", "test_parity.json"],
    "score": ["metrics.json", "results_tables.txt"],
    "future": ["outlook.csv", "outlook_exclusions.json"],
    "reload": ["reload_verification.json"],
    "report": [
        "forecasts.csv",
        "metrics.csv",
        "outlook.png",
        "failure_examples.png",
        "failure_examples.csv",
        "dhw_components.png",
        "environment.json",
        "run_summary.json",
        "conclusion.md",
        "README.txt",
        "checksums.json",
        "reef_evidence.zip",
    ],
}
# Planned splits and arms of each model-output file; checked against the frozen origins.
REEF_STAGE_ARMS = {
    "baselines": (("validation", "test"), ["persistence", "seasonal"]),
    "validation": (("validation",), ["chronos"]),
    "activity": (("validation",), ["chronos_180"]),
    "test": (("test",), ["chronos"]),
}
# Inputs whose identity the test stage requires to be unchanged since validation.
REEF_LOCKED_FILES = [
    "experiment_config.json",
    "origin_manifest.csv",
    "seasonal_reference.csv",
    "dataset_manifest.json",
    "model_manifest.json",
    "source_identity.json",
    "baselines_forecasts.csv",
    "validation_forecasts.csv",
    "activity_forecasts.csv",
]


def reef_invalidate(root: Path, stage: str) -> None:
    names = list(REEF_STAGES)
    for later in names[names.index(stage) :]:
        for name in REEF_STAGES[later]:
            (root / name).unlink(missing_ok=True)
            (root / name.replace(".csv", ".partial.csv")).unlink(missing_ok=True)


def reef_plan(root: Path) -> pd.DataFrame:
    """Frozen eligible origins, refused if the manifest changed after preparation."""
    config = reef_read(root / "experiment_config.json")
    if reef_sha(root / "origin_manifest.csv") != config["origin_manifest_sha256"]:
        raise ValueError("Origin manifest changed after freezing")
    plan = pd.read_csv(root / "origin_manifest.csv", parse_dates=["origin"])
    return plan[plan.eligible.astype(bool)].reset_index(drop=True)


def reef_stage_forecasts(root: Path, panel: pd.DataFrame, plan: pd.DataFrame, stage: str):
    splits, arms = REEF_STAGE_ARMS[stage]
    frame = reef_records(root / f"{stage}_forecasts.csv")
    try:
        validate_forecast_grid(frame, plan[plan.split.isin(splits)], arms, panel)
    except ValueError as exc:
        raise ValueError(f"{stage}_forecasts.csv: {exc}") from exc
    # Canonical key order: harmless row permutations cannot change floating-point sums.
    return frame.sort_values(FORECAST_KEYS, kind="stable").reset_index(drop=True)


def reef_prepare(root: Path) -> None:
    panel = reef_load_data(root)
    origins = origin_manifest(panel)
    origins.to_csv(root / "origin_manifest.csv", index=False)
    seasonal = fit_seasonal(panel)
    seasonal.to_csv(root / "seasonal_reference.csv", index=False)
    eligible = origins[origins.eligible]
    counts = eligible.groupby(["region_id", "split"]).size().unstack(fill_value=0)
    if (counts["test"] < 20).any():
        raise ValueError("Insufficient test origins; revise the design explicitly")
    reef_json(
        root / "experiment_config.json",
        {
            "context_days": 365,
            "activity_context_days": 180,
            "horizons": [7, 14, 28],
            "primary": "14-day HotSpot MAE; equal-region macro mean",
            "validation_years": [2023],
            "test_years": [2024, 2025],
            "seasonal_fit_years": [2017, 2022],
            "seed": 42,
            "availability": "retrospective zero-delay assumption; publication vintages unavailable",
            "pretraining_overlap": "not established absent; no unseen-pretraining claim",
            "origin_manifest_sha256": reef_sha(root / "origin_manifest.csv"),
        },
    )
    audit = {"regions": [], "checks": []}
    for region in panel.index.get_level_values(0).unique():
        frame = panel.xs(region)
        hs = frame.hotspot_c
        heat = hs.where(hs >= 1, 0).where(hs.notna())
        rebuilt = heat.rolling(84, min_periods=84).sum() / 7
        valid = rebuilt.notna() & frame.dhw_c_weeks.notna()
        error = float((rebuilt[valid] - frame.loc[valid, "dhw_c_weeks"]).abs().max())
        if error > 0.0001:
            raise ValueError(f"DHW contract disagreement in {region}")
        audit["regions"].append(
            {
                "region_id": region,
                "missing_days": int(hs.isna().sum()),
                "dhw_compared_days": int(valid.sum()),
                "dhw_max_error": error,
            }
        )
    # Refusal probes run before any model; invalid input must never become a success.
    sample = panel.xs("northern").reset_index()[["date", "hotspot_c"]].tail(400).copy()
    sample.insert(0, "region_id", "example")
    sample["date"] = sample["date"].dt.strftime("%Y-%m-%d")
    sample.to_csv(root / "byod_example.csv", index=False)
    for label, bad in [
        ("duplicate", pd.concat([sample, sample.iloc[-1:]])),
        ("negative", sample.assign(hotspot_c=-1)),
        ("nonfinite", sample.assign(hotspot_c=np.inf)),
    ]:
        try:
            validate_byod(bad, units="degC", semantics_confirmed=True)
        except ValueError:
            audit["checks"].append({"probe": label, "refused": True})
        else:
            raise AssertionError(f"Failed refusal probe: {label}")
    reef_json(root / "dataset_audit.json", audit)
    print(counts.to_string())
    print("DHW reconstructed within 0.0001 °C-weeks. Frozen origins and settings saved.")


def reef_experiment(root: Path, stage: str) -> None:
    start = time.perf_counter()
    panel = reef_load_data(root)
    manifest = origin_manifest(panel)
    config = reef_read(root / "experiment_config.json")
    if reef_sha(root / "origin_manifest.csv") != config["origin_manifest_sha256"]:
        raise ValueError("Origin manifest changed after freezing")
    seasonal = pd.read_csv(root / "seasonal_reference.csv")
    split = "validation" if stage in {"validation", "activity"} else "test"
    selected = manifest[manifest.eligible & (manifest.split == split)]
    if stage == "test":
        lock = reef_read(root / "experiment_lock.json")
        changed = [n for n in REEF_LOCKED_FILES if lock["sha256"].get(n) != reef_sha(root / n)]
        if changed:
            raise ValueError(f"Experiment changed after validation: {changed}")
    model = None
    if stage != "baselines":
        model = reef_load_model(reef_read(root / "model_manifest.json"), root / "model_cache")
    else:
        selected = manifest[manifest.eligible]
    output = []
    for number, row in enumerate(selected.itertuples(), 1):
        origin = pd.Timestamp(row.origin)
        length = 180 if stage == "activity" else 365
        context = context_at(panel, row.region_id, origin, length=length)
        values = np.asarray(context, dtype=float)
        if stage == "baselines":
            dates = pd.date_range(origin + pd.Timedelta(days=1), periods=28)
            predictions = baseline_forecasts(context, dates, seasonal, row.region_id)
            for arm, prediction in predictions.items():
                output.append(
                    forecast_frame(panel, row.region_id, origin, arm, prediction, row.split)
                )
        else:
            q = reef_predict(model, values)
            arm = "chronos_180" if stage == "activity" else "chronos"
            output.append(
                forecast_frame(panel, row.region_id, origin, arm, q[:, 1], row.split, quantiles=q)
            )
            if number == 1:
                reef_json(
                    root / f"{stage}_parity.json",
                    {"history": values.tolist(), "quantiles": q.tolist()},
                )
        # Write progress immediately so an interrupted run never fabricates completion.
        pd.concat(output, ignore_index=True).to_csv(
            root / f"{stage}_forecasts.partial.csv", index=False
        )
        if number % 25 == 0:
            print(f"{stage}: {number}/{len(selected)} origins", flush=True)
    result = pd.concat(output, ignore_index=True)
    result.to_csv(root / f"{stage}_forecasts.csv", index=False)
    summary = {
        "stage": stage,
        "origins": len(selected),
        "seconds": time.perf_counter() - start,
        "complete": True,
        "rows": len(result),
    }
    if "quantile_crossing_c" in result:
        crossed = result.quantile_crossing_c > 0
        summary["quantile_crossing_days"] = int(crossed.sum())
        summary["quantile_crossing_max_c"] = float(result.quantile_crossing_c.max())
    if model is not None:
        import torch

        summary["peak_gpu_bytes"] = torch.cuda.max_memory_allocated()
    reef_json(root / f"{stage}_summary.json", summary)
    print(summary)


def reef_lock(root: Path) -> None:
    validation = reef_read(root / "validation_summary.json")
    activity = reef_read(root / "activity_summary.json")
    if not validation["complete"] or not activity["complete"]:
        raise ValueError("Validation and controlled experiment must finish first")
    setup = reef_read(root / "setup_summary.json")
    baseline = reef_read(root / "baselines_summary.json")
    elapsed = setup["seconds"] + baseline["seconds"] + validation["seconds"] + activity["seconds"]
    # Conservative model-stage estimate includes test origins and two extra reloads.
    remaining = validation["seconds"] * (184 + 5) / validation["origins"] + 120
    peak = max(validation.get("peak_gpu_bytes", 0), activity.get("peak_gpu_bytes", 0))
    budget = {
        "elapsed_seconds": elapsed,
        "estimated_remaining_seconds": remaining,
        "estimated_total_seconds": elapsed + remaining,
        "peak_gpu_bytes": peak,
        "limit_seconds": 1800,
        "limit_gpu_bytes": 12 * 1024**3,
        "scope": "validation-based estimate, not measured complete runtime",
    }
    reef_json(root / "budget_check.json", budget)
    if elapsed + remaining > 1800 or peak > 12 * 1024**3:
        raise RuntimeError(
            "Validation exceeds the resource budget; preserve logs and revise design before test"
        )
    panel, plan = reef_load_data(root), reef_plan(root)
    for stage in ("baselines", "validation", "activity"):
        reef_stage_forecasts(root, panel, plan, stage)
    reef_json(
        root / "experiment_lock.json",
        {
            "sha256": {name: reef_sha(root / name) for name in REEF_LOCKED_FILES},
            "selection": "365-day default retained; no test-based tuning",
        },
    )
    print("365-day context retained. Test settings locked.")


def reef_outlook_rows(region, origin, arm, context, raw, quantiles=None) -> list[dict]:
    """Unscored forward rows keep raw, constrained, quantile and DHW-component fields."""
    raw = np.asarray(raw, dtype=float)
    crossing = None
    if quantiles is not None:
        model_q = np.asarray(quantiles, dtype=float)
        quantiles, crossing = rearrange_quantiles(model_q)
        raw = quantiles[:, 1]
    components = compose_dhw(context, np.maximum(raw, 0))
    dates = pd.date_range(pd.Timestamp(origin) + pd.Timedelta(days=1), periods=len(raw))
    rows = []
    for k, day in enumerate(dates):
        row = {
            "region_id": region,
            "origin": pd.Timestamp(origin),
            "target_date": day,
            "lead": k + 1,
            "arm": arm,
            "raw_hotspot_c": float(raw[k]),
            "hotspot_c": float(max(0.0, raw[k])),
            "clipped_to_zero": bool(raw[k] < 0),
            "dhw_c_weeks": components["dhw"][k],
            "known_dhw": components["known"][k],
            "predicted_dhw": components["predicted"][k],
        }
        for i, label in enumerate(["q10", "q50", "q90"]):
            row["model_" + label] = float(model_q[k, i]) if crossing is not None else np.nan
        row["quantile_crossing_c"] = float(crossing[k]) if crossing is not None else np.nan
        for i, label in enumerate(["q10", "q50", "q90"]):
            value = float(quantiles[k, i]) if quantiles is not None else np.nan
            row["raw_" + label] = value
            row[label] = max(0.0, value) if quantiles is not None else np.nan
        rows.append(row)
    return rows


def reef_future(root: Path) -> None:
    panel = reef_load_data(root)
    origin = pd.Timestamp(reef_read(root / "dataset_manifest.json")["coverage_end"])
    seasonal = pd.read_csv(root / "seasonal_reference.csv")
    model = reef_load_model(reef_read(root / "model_manifest.json"), root / "model_cache")
    rows, skipped = [], []
    for region in panel.index.get_level_values(0).unique():
        try:
            context = context_at(panel, region, origin)
        except ValueError as exc:
            skipped.append({"region_id": region, "reason": str(exc)})
            continue
        dates = pd.date_range(origin + pd.Timedelta(days=1), periods=28)
        arms = baseline_forecasts(context, dates, seasonal, region)
        q = reef_predict(model, context)
        arms["chronos"] = q[:, 1]
        for arm, raw in arms.items():
            quantiles = q if arm == "chronos" else None
            rows.extend(reef_outlook_rows(region, origin, arm, context, raw, quantiles))
    pd.DataFrame(rows).to_csv(root / "outlook.csv", index=False)
    reef_json(root / "outlook_exclusions.json", skipped)
    print(
        f"Illustrative origin {origin.date()}; {len(skipped)} regions excluded for missing context."
    )


def reef_same(saved: pd.DataFrame, again: pd.DataFrame) -> None:
    """Numeric parity plus identity: region, arm, split and calendar dates must match."""
    if len(saved) != len(again):
        raise AssertionError("Reloaded row count differs")
    for col in ["region_id", "arm", "split", "origin", "target_date"]:
        if not (saved[col].astype(str).to_numpy() == again[col].astype(str).to_numpy()).all():
            raise AssertionError(f"Reloaded identity column differs: {col}")
    for col in again.select_dtypes(include="number"):
        np.testing.assert_allclose(saved[col], again[col], atol=1e-8, rtol=1e-7, equal_nan=True)


def reef_reload(root: Path) -> None:
    """Fresh CLI process recomputes outputs and reloads real model for one origin."""
    panel, plan = reef_load_data(root), reef_plan(root)
    seasonal = pd.read_csv(root / "seasonal_reference.csv")
    baseline = reef_stage_forecasts(root, panel, plan, "baselines")
    rebuilt = []
    for (region, origin), group in baseline.groupby(["region_id", "origin"]):
        context = context_at(panel, region, origin)
        dates = pd.date_range(origin + pd.Timedelta(days=1), periods=28)
        for arm, pred in baseline_forecasts(context, dates, seasonal, region).items():
            rebuilt.append(forecast_frame(panel, region, origin, arm, pred, group.iloc[0]["split"]))
    recomputed = pd.concat(rebuilt, ignore_index=True)
    keys = ["region_id", "origin", "arm", "lead"]
    original = baseline.sort_values(keys).reset_index(drop=True)
    recomputed = recomputed.sort_values(keys).reset_index(drop=True)
    reef_same(original, recomputed)
    # Recalculate model-derived DHW and metrics from serialized raw predictions.
    for stage in ("validation", "activity", "test"):
        saved = reef_stage_forecasts(root, panel, plan, stage)
        rebuilt_model = []
        for (region, origin, arm), group in saved.groupby(["region_id", "origin", "arm"]):
            group = group.sort_values("lead")
            # Rebuild from the model's own (unsorted) quantiles, as the stage did.
            qcols = ["model_q10", "model_q50", "model_q90"]
            qs = group[qcols].to_numpy() if set(qcols).issubset(group) else None
            raw_col = "model_q50" if qs is not None else "raw_hotspot_c"
            rebuilt_model.append(
                forecast_frame(
                    panel,
                    region,
                    origin,
                    arm,
                    group[raw_col].to_numpy(),
                    group.iloc[0]["split"],
                    quantiles=qs,
                )
            )
        again = pd.concat(rebuilt_model, ignore_index=True).sort_values(keys).reset_index(drop=True)
        reef_same(saved.sort_values(keys).reset_index(drop=True), again)
    if reef_read(root / "metrics.json") != reef_metrics(root):
        raise AssertionError("Fresh-process metric reconstruction mismatch")
    sample = reef_read(root / "validation_parity.json")
    model = reef_load_model(reef_read(root / "model_manifest.json"), root / "model_cache")
    q = reef_predict(model, sample["history"])
    np.testing.assert_allclose(q, sample["quantiles"], atol=1e-5, rtol=1e-5)
    reef_json(
        root / "reload_verification.json",
        {
            "fresh_process": True,
            "baseline_parity": True,
            "derived_dhw_parity": True,
            "model_reload_parity": True,
            "metrics_reload_parity": True,
            "atol": 1e-5,
            "rtol": 1e-5,
        },
    )
    print("Fresh-process baseline, DHW and real-model reload parity passed.")


def reef_joined(root: Path, stages=("baselines", "validation", "activity", "test")):
    panel, plan = reef_load_data(root), reef_plan(root)
    frames = [reef_stage_forecasts(root, panel, plan, stage) for stage in stages]
    return pd.concat(frames, ignore_index=True), plan


REEF_SCOPES = {"all": None, "common_2024": 2024, "available_2025": 2025}


def reef_check_support(report: dict, plan: pd.DataFrame, year: int | None) -> None:
    """Recomputed macro support must equal the frozen plan; nothing is silently dropped."""
    if year is not None:
        plan = plan[plan.origin.dt.year == year]
    for row in report["macro"]:
        subset = plan[plan.split == row["split"]]
        if row["period"] != "full":
            subset = subset[subset.origin.dt.year == int(row["period"])]
        expected = (int(subset.region_id.nunique()), len(subset))
        if (row["regions"], row["origins"]) != expected:
            raise ValueError(
                f"Support mismatch for {row['split']}/{row['period']}/{row['arm']}/"
                f"{row['horizon']}: {(row['regions'], row['origins'])} != planned {expected}"
            )


def reef_metrics(root: Path) -> dict:
    joined, plan = reef_joined(root)
    evaluate_forecasts(
        joined[joined.split == "test"], expected_arms=["persistence", "seasonal", "chronos"]
    )
    reports = {}
    for label, year in REEF_SCOPES.items():
        subset = joined if year is None else joined[joined.origin.dt.year == year]
        reports[label] = evaluate_forecasts(subset)
        reef_check_support(reports[label], plan, year)
    return reports


def reef_score(root: Path) -> None:
    reports = reef_metrics(root)
    reef_json(root / "metrics.json", reports)
    text = reef_results_tables(reports)
    (root / "results_tables.txt").write_text(text, encoding="utf-8")
    print(text)
    print("Saved metrics for independent fresh-process reconstruction.")


def _fmt(frame: pd.DataFrame) -> str:
    return frame.to_string(index=False, float_format=lambda v: f"{v:.3f}", na_rep="n/a")


def _event_reason(n: int, support: int, predicted: int) -> str:
    if n == 0:
        return "no observations"
    if support == 0 and predicted == 0:
        return "no observed or predicted positives"
    if support == 0:
        return "no observed positives: recall undefined"
    if predicted == 0:
        return "no predicted positives: precision undefined"
    return ""


def reef_event_table(records: list[dict], horizons=(14, 28)) -> pd.DataFrame:
    """Pool region-level threshold counts per arm; undefined ratios stay unavailable."""
    frame = pd.DataFrame(records)
    frame = frame[frame.horizon.isin(horizons)]
    keys = ["horizon", "threshold", "subset", "arm"]
    pooled = frame.groupby(keys, as_index=False)[["n", "tp", "fp", "fn", "tn"]].sum()
    pooled["observed_pos"] = pooled.tp + pooled.fn
    predicted = pooled.tp + pooled.fp
    pooled["precision"] = (pooled.tp / predicted).where(predicted > 0)
    pooled["recall"] = (pooled.tp / pooled.observed_pos).where(pooled.observed_pos > 0)
    pooled["threshold"] = pooled.threshold.map(lambda t: f"{t:g} °C-wk")
    pooled["note"] = [
        _event_reason(int(n), int(s), int(p))
        for n, s, p in zip(pooled.n, pooled.observed_pos, predicted, strict=True)
    ]
    return pooled


def reef_results_tables(reports: dict) -> str:
    """Compact learner views of the frozen comparison; complete tables are in metrics.csv."""
    report = reports["all"]
    macro = pd.DataFrame(report["macro"])
    test = macro[macro.split == "test"]
    names = {"full": "Full test (unbalanced)", "2024": "Common 2024", "2025": "2025 (3 regions)"}
    out = []
    rows = []
    for period, label in names.items():
        for arm in ["persistence", "seasonal", "chronos", "dhw_persistence"]:
            sel = test[(test.period == period) & (test.arm == arm)].set_index("horizon")
            if sel.empty:
                continue
            rows.append(
                {
                    "panel": label,
                    "arm": arm,
                    "MAE 7d": sel.hotspot_mae.get(7, np.nan),
                    "MAE 14d": sel.hotspot_mae.get(14, np.nan),
                    "MAE 28d": sel.hotspot_mae.get(28, np.nan),
                    "DHW MAE 14d": sel.dhw_endpoint_mae.get(14, np.nan),
                    "DHW MAE 28d": sel.dhw_endpoint_mae.get(28, np.nan),
                    "regions": int(sel.regions.get(14)),
                    "origins": int(sel.origins.get(14)),
                    "days@14d": int(sel.days.get(14)),
                }
            )
    out += [
        "A. Test HotSpot MAE (°C) and DHW endpoint MAE (°C-weeks); equal-region macro means.",
        "   Support counts show exactly which regions and origins each score covers.",
        "   dhw_persistence carries DHW forward and has no HotSpot score (n/a).",
        _fmt(pd.DataFrame(rows)),
        "",
    ]
    region = pd.DataFrame(report["metrics"])
    region = region[(region.split == "test") & (region.period == "full") & (region.horizon == 14)]
    wide = region[region.arm != "dhw_persistence"].pivot(
        index="region_id", columns="arm", values="hotspot_mae"
    )
    wide["origins"] = region.groupby("region_id").origins.first()
    out += [
        "B. 14-day HotSpot MAE by region, full test panel (°C).",
        _fmt(wide.reset_index()),
        "",
    ]
    chronos = test[test.arm == "chronos"]
    unc = chronos[["period", "horizon", "coverage_10_90", "width_10_90"]].copy()
    for col in ["pinball_q10", "pinball_q50", "pinball_q90"]:
        unc[col] = chronos[col]
    out += [
        "C. Chronos q10–q90 marginal interval: coverage (nominal 0.80), width and pinball loss.",
        _fmt(unc.sort_values(["period", "horizon"])),
        "",
    ]
    metrics = pd.DataFrame(report["metrics"])
    hs = metrics[(metrics.split == "test") & (metrics.period == "full")]
    hs = hs[hs.arm != "dhw_persistence"]
    hs = hs.groupby(["arm", "horizon"], as_index=False).agg(
        high_stress_days=("high_stress_days", "sum"), high_stress_mae=("high_stress_mae", "mean")
    )
    hs["high_stress_days"] = hs.high_stress_days.astype(int)
    out += [
        "D. Errors on observed high-stress days (HotSpot >= 1 °C), full test panel.",
        "   high_stress_mae is the equal-region mean over regions with such days.",
        _fmt(hs),
        "",
    ]
    thresholds = [r for r in report["thresholds"] if r["split"] == "test" and r["period"] == "full"]
    events = reef_event_table(thresholds)
    out += [
        "E. DHW threshold events at the endpoint, pooled over region-origins (full test panel).",
        "   new_exceedance keeps origins that were below the threshold at the origin.",
        "   Unavailable precision/recall is n/a with a reason; it is never perfect performance.",
        _fmt(events),
        "",
    ]
    paired = pd.DataFrame(report["paired_differences"])
    paired = paired[(paired.split == "test") & (paired.horizon == 14)]
    summary = (
        paired.groupby(["period", "baseline"])
        .mae_difference.agg(
            pairs="count",
            mean_difference="mean",
            median_difference="median",
            chronos_better=lambda d: float((d < 0).mean()),
            tie=lambda d: float((d == 0).mean()),
            chronos_worse=lambda d: float((d > 0).mean()),
        )
        .reset_index()
    )
    out += [
        "F. Paired 14-day MAE difference, Chronos minus baseline, per region-origin (°C).",
        "   Negative values favour Chronos. The last three columns are shares of region-origins",
        "   where Chronos was better, tied or worse. Descriptive only; no significance test.",
        _fmt(summary),
        "",
        "Complete tables: metrics.csv (scope/table columns) inside reef_evidence.zip.",
    ]
    return "\n".join(out)


def reef_compare(root: Path) -> None:
    """Validation-only view of the controlled context-length activity, before the test."""
    joined, plan = reef_joined(root, ("baselines", "validation", "activity"))
    validation = joined[joined.split == "validation"]
    report = evaluate_forecasts(validation)
    reef_check_support(report, plan[plan.split == "validation"], None)
    macro = pd.DataFrame(report["macro"])
    cols = ["arm", "horizon", "hotspot_mae", "dhw_endpoint_mae", "regions", "origins", "days"]
    table = macro[cols].sort_values(["horizon", "arm"])
    region = pd.DataFrame(report["metrics"])
    region = region[(region.horizon == 14) & region.arm.isin(["chronos", "chronos_180"])]
    wide = region.pivot(index="region_id", columns="arm", values="hotspot_mae")
    wide["180 minus 365"] = wide.chronos_180 - wide.chronos
    wide["origins"] = region.groupby("region_id").origins.first()
    manifest = pd.read_csv(root / "origin_manifest.csv", parse_dates=["origin"])
    valid = plan[plan.split == "validation"].iloc[0]
    origin = pd.Timestamp(valid.origin)
    rejected = manifest[~manifest.eligible.astype(bool)]
    preferred = rejected[rejected.reason.str.contains("incomplete_365_day_context")]
    reject = (preferred if len(preferred) else rejected).iloc[0]
    lines = [
        "Validation 2023 only; the test set is still unopened.",
        "Macro HotSpot MAE (°C) and DHW endpoint MAE (°C-weeks) with support:",
        _fmt(table),
        "",
        "14-day HotSpot MAE by region: 365-day default (chronos) vs 180-day activity:",
        _fmt(wide.reset_index()),
        "",
        f"Trace a valid context: {valid.region_id}, origin {origin.date()}.",
        f"  365-day context {(origin - pd.Timedelta(days=364)).date()} to {origin.date()};",
        f"  180-day context {(origin - pd.Timedelta(days=179)).date()} to {origin.date()}"
        " (its final 180 days);",
        f"  targets {(origin + pd.Timedelta(days=1)).date()} to "
        f"{(origin + pd.Timedelta(days=28)).date()}.",
        f"Trace a rejected origin: {reject.region_id}, {pd.Timestamp(reject.origin).date()}: "
        f"{reject.reason}.",
        "The 365-day test default stays fixed whichever validation context wins.",
    ]
    text = "\n".join(lines)
    (root / "validation_tables.txt").write_text(text, encoding="utf-8")
    print(text)


def reef_report(root: Path) -> None:
    import importlib.metadata

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt

    stages = ["baselines", "validation", "activity", "test"]
    joined, _plan = reef_joined(root)
    evaluate_forecasts(
        joined[joined.split == "test"], expected_arms=["persistence", "seasonal", "chronos"]
    )
    joined.to_csv(root / "forecasts.csv", index=False)
    reports = reef_read(root / "metrics.json")
    if reports != reef_metrics(root):
        raise AssertionError("Metrics changed after scoring")
    flat = []
    for scope, report in reports.items():
        for table, records in report.items():
            if isinstance(records, list):
                for row in records:
                    if isinstance(row, dict):
                        flat.append({"scope": scope, "table": table, **row})
    pd.DataFrame(flat).to_csv(root / "metrics.csv", index=False)
    outlook = reef_records(root / "outlook.csv")
    regions = list(outlook.region_id.unique())
    issued = pd.Timestamp(outlook.origin.iloc[0]).date()
    excluded = reef_read(root / "outlook_exclusions.json")

    def dates_axis(ax):
        locator = mdates.AutoDateLocator(minticks=3, maxticks=7)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

    fig, axes = plt.subplots(len(regions), 1, figsize=(10, 3 * len(regions)), squeeze=False)
    for ax, region in zip(axes[:, 0], regions, strict=True):
        for arm, group in outlook[outlook.region_id == region].groupby("arm"):
            ax.plot(group.target_date, group.dhw_c_weeks, label=arm)
        ax.axhline(4, color="orange", linestyle=":")
        ax.axhline(8, color="red", linestyle=":")
        ax.set_title(f"{region.title()} — DHW from frozen origin {issued}")
        ax.set_ylabel("°C-weeks")
        dates_axis(ax)
        ax.legend()
    note = "Illustrative retrospective outlook issued from frozen origin " + str(issued)
    note += "; not a live warning."
    if excluded:
        note += "\nNot shown (incomplete 365-day context): " + ", ".join(
            e["region_id"] for e in excluded
        )
    fig.suptitle(note, fontsize=10)
    fig.tight_layout()
    fig.savefig(root / "outlook.png", dpi=140)
    plt.close(fig)
    # Predeclared failure selection: worst 14-day Chronos MAE in each region.
    test = joined[(joined.split == "test") & (joined.arm == "chronos")]
    errors = test[test.lead <= 14].copy()
    errors["absolute_error"] = abs(errors.hotspot_c - errors.actual_hotspot_c)
    ranked = errors.groupby(["region_id", "origin"]).absolute_error.mean().reset_index()
    chosen = ranked.sort_values(
        ["absolute_error", "origin"], ascending=[False, True]
    ).drop_duplicates("region_id")
    chosen.to_csv(root / "failure_examples.csv", index=False)
    fig, axes = plt.subplots(len(chosen), 1, figsize=(10, 3 * len(chosen)), squeeze=False)
    for ax, row in zip(axes[:, 0], chosen.itertuples(), strict=True):
        group = test[(test.region_id == row.region_id) & (test.origin == row.origin)]
        ax.plot(group.target_date, group.actual_hotspot_c, label="Observed HotSpot")
        ax.plot(group.target_date, group.hotspot_c, label="Chronos median")
        ax.fill_between(group.target_date, group.q10, group.q90, alpha=0.2, label="q10–q90")
        ax.set_title(f"{row.region_id}: worst 14-day MAE origin {row.origin.date()}")
        ax.set_ylabel("HotSpot °C")
        dates_axis(ax)
        ax.legend()
    fig.tight_layout()
    fig.savefig(root / "failure_examples.png", dpi=140)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(10, 4))
    sample = outlook[(outlook.region_id == regions[0]) & (outlook.arm == "chronos")]
    ax.stackplot(
        sample.target_date,
        sample.known_dhw,
        sample.predicted_dhw,
        labels=["Known historical contribution", "New predicted contribution"],
        alpha=0.7,
    )
    ax.set_title(f"{regions[0].title()}: what contributes to DHW after frozen origin {issued}?")
    ax.set_ylabel("°C-weeks")
    dates_axis(ax)
    ax.legend()
    fig.tight_layout()
    fig.savefig(root / "dhw_components.png", dpi=140)
    plt.close(fig)
    env = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": {
            name: importlib.metadata.version(name)
            for name in ["torch", "chronos-forecasting", "numpy", "pandas", "transformers"]
        },
    }
    reef_json(root / "environment.json", env)
    reload = reef_read(root / "reload_verification.json")
    summary = {
        "status": "executed_candidate_requires_maintainer_review",
        "scope": "retrospective regional satellite heat stress; not observed bleaching",
        "stages": {s: reef_read(root / f"{s}_summary.json") for s in stages},
        "reload": reload,
        "refusal_probes": reef_read(root / "dataset_audit.json")["checks"],
        "negative_point_predictions_clamped": int((joined.raw_hotspot_c < 0).sum()),
        "quantile_crossing_days": int((joined.quantile_crossing_c.fillna(0) > 0).sum()),
        "quantile_crossing_max_c": float(joined.quantile_crossing_c.fillna(0).max()),
        "source": reef_read(root / "source_identity.json"),
        "model": reef_read(root / "model_manifest.json"),
        "dataset_archive_sha256": reef_read(root / "dataset_manifest.json")["archive"]["sha256"],
        "warnings": [
            "Western/Southern have no eligible 2025 test origins",
            "No confirmed exclusion from foundation-model pretraining",
            "Marginal HotSpot bands are not DHW probability intervals",
            "Crossing Chronos quantiles are sorted (monotone rearrangement) before scoring",
        ],
    }
    reef_json(root / "run_summary.json", summary)
    (root / "conclusion.md").write_text(
        "# Interpretation\n\nUse metrics.csv: compare 14-day HotSpot MAE on identical origins. "
        "Report the winning arm and error, persistence/seasonal errors, common-2024 comparison, "
        "and the sign of the paired difference. A loss or inconclusive result is valid.\n\n"
        "Explain one failure mode, interval coverage and event support. Separate already "
        "accumulated DHW from newly predicted stress. These are regional satellite summaries, "
        "not reef-level observations or bleaching diagnoses. Missing contexts exclude two "
        "regions in 2025; historical availability and pretraining independence are unresolved.\n\n"
        "Personal completion record: task, evidence, baseline, limitation, next experiment. "
        "This record is not a required submission.\n",
        encoding="utf-8",
    )
    (root / "README.txt").write_text(
        "NOAA CRW public-domain data, credited in dataset_manifest.json. "
        "This bundle contains experiment outputs, not model weights. Model identity is pinned. "
        "Candidate evidence requires maintainer review; it is not an operational NOAA product.\n"
    )
    keep = [
        p
        for p in root.iterdir()
        if p.is_file()
        and p.suffix in {".json", ".csv", ".png", ".md", ".txt"}
        and ".partial." not in p.name
        and p.name != "byod_example.csv"
        and p.name != "checksums.json"
    ]
    reef_json(root / "checksums.json", {p.name: reef_sha(p) for p in sorted(keep)})
    bundle = root / "reef_evidence.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for p in keep + [root / "checksums.json"]:
            archive.write(p, p.name)
    print(f"Exported {bundle}. Results are evidence, not release approval.")


def reef_byod(root: Path, source: Path) -> Path:
    """Forward-only BYOD: literal IDs, unscored forecasts and a per-input receipt."""
    raw = source.read_bytes()
    digest = hashlib.sha256(raw).hexdigest()
    frame = validate_byod(read_byod_csv(source), units="degC", semantics_confirmed=True)
    panel = daily_panel([frame])
    manifest = reef_read(root / "model_manifest.json")
    out = root / "byod" / digest[:16]
    out.mkdir(parents=True, exist_ok=True)
    for stale in ("forecasts.csv", "receipt.json"):
        (out / stale).unlink(missing_ok=True)
    model = reef_load_model(manifest, root / "model_cache")
    rows, inputs = [], []
    for region in panel.index.get_level_values(0).unique():
        origin = panel.xs(region).index.max()
        history = context_at(panel, region, origin)
        q = reef_predict(model, history)
        rows.extend(reef_outlook_rows(region, origin, "chronos", history, q[:, 1], q))
        inputs.append(
            {
                "region_id": region,
                "first_date": str(panel.xs(region).index.min().date()),
                "origin": str(origin.date()),
                "context_days": len(history),
            }
        )
    forecasts = pd.DataFrame(rows)
    forecasts.to_csv(out / "forecasts.csv", index=False)
    identity = root / "source_identity.json"
    reef_json(
        out / "receipt.json",
        {
            "input_file": source.name,
            "input_sha256": digest,
            "input_rows": len(frame),
            "regions": inputs,
            "units": "degC",
            "semantics": "NOAA-compatible threshold-relative HotSpot, confirmed by user",
            "model": {"modelId": manifest["modelId"], "revision": manifest["revision"]},
            "runner_sha256": (
                reef_read(identity).get("runner_sha256") if identity.exists() else None
            ),
            "outputs": {"forecasts.csv": reef_sha(out / "forecasts.csv")},
            "scored": False,
            "scope": "forward inference only; no future outcomes, no accuracy claim",
        },
    )
    print(f"{len(inputs)} BYOD region(s): {[i['region_id'] for i in inputs]}")
    print(f"Saved unscored forecasts and receipt in {out}")
    return out


def reef_main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "stage",
        choices=[
            *REEF_STAGES,
            "byod",
        ],
    )
    parser.add_argument("--root", type=Path, required=True)
    parser.add_argument("--csv", type=Path)
    args = parser.parse_args()
    root = args.root.resolve()
    if args.stage == "byod":
        reef_byod(root, args.csv)
        return
    reef_invalidate(root, args.stage)
    if args.stage in {"baselines", "validation", "activity", "test"}:
        reef_experiment(root, args.stage)
    else:
        {
            "prepare": reef_prepare,
            "score": reef_score,
            "compare": reef_compare,
            "lock": reef_lock,
            "future": reef_future,
            "reload": reef_reload,
            "report": reef_report,
        }[args.stage](root)


if __name__ == "__main__":
    reef_main()
'''

In [ ]:
RUNNER_SOURCE = "from __future__ import annotations\n" + CORE_SOURCE + MODEL_SOURCE + RUNTIME_SOURCE
compile(RUNNER_SOURCE, "reef_runner.py", "exec")
(RUN_ROOT / "reef_runner.py").write_text(RUNNER_SOURCE, encoding="utf-8")
SOURCE_IDENTITY = {'builder': 'tools/build_reef_capstone_notebook.py', 'source_sha256': {'reef_core.py': '87d7e6c8d3ce1340e2806c326a1afd2c5a6df58729947f490967ebc593b1ee0b', 'reef_models.py': '508526d5fdf959b89f2c9fd826baae32b9c5814b5a891cc46f3ce02533f22714', 'reef_runtime.py': '64e703db6c8efd5e75e9f57f1f6df1accee03e85eeb7999478e7c2058166da6d'}, 'requirements_sha256': '83c201033d0f71b7a94dad88852b6edbacba0b37b1d67895e5c32ec92d3b90e5', 'dataset_sha256': '320a04998ff0f0ce4a66264fc83b585c0fafd653844d533468d98a84d6d07b43', 'spec_version': '2.2'}

SOURCE_IDENTITY["runner_sha256"] = hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest()
(RUN_ROOT / "source_identity.json").write_text(json.dumps(SOURCE_IDENTITY, indent=2))
def run_stage(name, *extra):
    print(f"[{time.strftime('%H:%M:%S')}] stage {name} started", flush=True)
    command = [PYTHON, RUN_ROOT / "reef_runner.py", name, "--root", RUN_ROOT, *extra]
    seconds = run_logged(command, f"stage_{name}")
    print(f"[{time.strftime('%H:%M:%S')}] stage {name} finished in {seconds:.0f} s", flush=True)
run_stage("prepare")

**What to notice:** DHW should reconstruct to within 0.0001 °C-weeks. The origin table should show
23 validation origins per region, 46 test origins for Northern/Central/Eastern, and 23 for Western/Southern.
The latter two have no eligible 2025 origins because their 365-day contexts include gaps.
This loss of coverage is part of the result, not a model failure.

## 4. Evaluation practice: freeze the calendar

Fit the seasonal reference on 2017–2022 only. Use 2023 for validation; test on 2024–2025. Origins are the
1st and 15th of each month, excluding forecasts that cross the year boundary. All arms share identical eligible
origins and 365-day contexts. Score 7-, 14- and 28-day prefixes of the same 28-day prediction.

The primary outcome is equal-region mean 14-day HotSpot MAE (mean absolute error, °C). Show the complete-panel
2024 result alongside the unbalanced full test and three-region 2025 result. Foundation-model pretraining
overlap is unresolved; this is not a claim of a pristine unseen-training benchmark. Reprocessed historical
observations do not prove what was published at each historical origin.

**Predict:** why is randomly shuffling daily observations an invalid forecast experiment?
<details><summary>Show a sample answer</summary>It allows future conditions into development and ignores
the information that would have been available at the forecast origin. Date-aware splits preserve that boundary.</details>

In [ ]:
run_stage("baselines")
print("Baseline forecasts saved. Both arms use exactly the planned contexts and outcomes.")

## 5. Compare a foundation model

**Persistence** repeats the last HotSpot. **Seasonal reference** repeats the development-period mean for each
month/day. **Chronos-2** forecasts from recent history without fitting new weights. Its raw negative predictions
are saved, then constrained to zero consistently with the nonnegative target. Quantile bands are marginal
daily forecasts, not simultaneous guarantees.

**Predict:** will recent history outperform seasonal averages during unusual heat? The experiment may say no.
Run validation first. Model downloads are pinned by immutable revision and verified digests.

In [ ]:
run_stage("validation")

## 6. Change one thing: how much history?

Predict whether shortening context from 365 to 180 days helps. The next cell changes only context length on the
same validation origins; model, dates, horizon and scoring remain fixed. It then prints a validation-only table:
compare the `chronos_180` and `chronos` rows at 7, 14 and 28 days, check the support columns (regions, origins,
days), and read the traced valid and rejected contexts. The 365-day test default remains frozen regardless of
this activity.

<details><summary>How to interpret the comparison</summary>A shorter context may respond to recent conditions,
but can lose seasonal information. Use measured errors; a plausible explanation alone is not evidence.</details>

In [ ]:
run_stage("activity")
run_stage("compare")
run_stage("lock")

## 7. Locked test and accumulated stress

Now evaluate once on the held-out origins. For each arm, future DHW combines known past heat with that arm's
predicted HotSpots. The separate DHW-persistence reference simply carries DHW at the origin forward.
Good DHW scores may reflect already accumulated heat; check HotSpot skill and the known/predicted contribution.
The DHW of median HotSpots is not necessarily the median DHW forecast. No DHW probability bands are claimed.

Scoring prints six compact tables, labelled A–F: (A) the full, common-2024 and three-region-2025 comparisons with
region/origin/day support; (B) regional 14-day errors; (C) interval coverage, width and pinball loss;
(D) high-stress-day errors; (E) 4 and 8 °C-week threshold events, including new exceedances; (F) paired
Chronos-minus-baseline differences. Section 10 asks questions you can answer from these tables.

In [ ]:
run_stage("test")
run_stage("score")

## 8. Illustrative regional outlook

Forecast from the frozen snapshot end, 2026-09-26. These dates are explicit: this is not a live warning.
Regions without a complete recent context are listed as excluded; no gap filling is hidden in the chart.
Western and Southern Philippines are expected to be excluded here: their last 365 days contain missing dates.
The outlook file keeps raw and constrained HotSpot, Chronos quantiles and the known/new DHW contributions.
Numerical DHW indicators at 4 and 8 °C-weeks summarize thermal exposure, not field-observed bleaching.

In [ ]:
run_stage("future")

## 9. Export and independent verification

A fresh process recomputes baseline and DHW values from serialized inputs, then reloads the pinned model for
one origin. Prediction parity is checked with declared tolerances. The bundle records dataset and model identity,
environment, origin exclusions, metrics, forecasts and evidence. A successful run is evidence for maintainer review,
not automatic release approval.

In [ ]:
run_stage("reload")
run_stage("report")

In [ ]:
# Read reports with the standard library; no notebook-kernel dependency changes.
reports = json.loads((RUN_ROOT / "metrics.json").read_text())
from IPython.display import display, Markdown, Image, FileLink
rows = reports["all"]["macro"]
keys = ["split", "period", "arm", "hotspot_mae", "dhw_endpoint_mae", "regions", "origins", "days"]
table = ("Primary 14-day summary with support. Tables A–F printed by the scoring cell hold the detail.\n\n"
         "| Split | Period | Arm | HotSpot MAE | DHW endpoint MAE | Regions | Origins | Days |\n"
         "|---|---|---|---:|---:|---:|---:|---:|\n")
def cell(value):
    return "n/a" if value is None else f"{value:.3f}" if isinstance(value, float) else str(value)
for row in rows:
    if row["horizon"] == 14:
        table += "| " + " | ".join(cell(row.get(key)) for key in keys) + " |\n"
display(Markdown(table))
display(Image(filename=str(RUN_ROOT / "outlook.png")))
display(Image(filename=str(RUN_ROOT / "dhw_components.png")))
display(Image(filename=str(RUN_ROOT / "failure_examples.png")))
display(FileLink(str(RUN_ROOT / "reef_evidence.zip")))
print("All mandatory stages completed. Preserve this executed notebook and the evidence bundle.")

## 10. Interpret the evidence

- Which arm has the lowest 14-day HotSpot MAE (table A)? How large is its paired difference from persistence
  and seasonality (table F)? Check the regions/origins columns so both scores cover the same cases.
- Does the common-2024 comparison support the same conclusion as the unbalanced full panel (table A)?
- Is good DHW performance mainly inherited from known history (compare `dhw_persistence` in table A and the
  component chart)? Does the model anticipate new exceedances (table E, `new_exceedance`)?
- Inspect q10–q90 coverage and width (table C) and high-stress-day errors (table D). An 80% nominal band need not
  cover 80% here.
- Report event support (table E). "No observations", "no observed positives" and a poor score are different
  findings; undefined precision/recall stays n/a with a reason and never becomes perfect performance.

<details><summary>Conclusion template</summary>For [regions and dates], [arm] had [MAE] at 14 days versus
[baseline errors]. The common-period comparison [agreed/differed]. A failure case was [example]. These findings
describe a retrospective satellite stress indicator; they do not establish reef-level bleaching, operational
warning skill, or independence from foundation-model training data. My next controlled experiment is [change].</details>

Your completion record is a personal learning aid, **not a required submission**. An inconclusive result is acceptable.

## 11. Optional: bring your own daily HotSpot data

Disabled by default. Supply `region_id,date,hotspot_c` with ISO dates, °C units, and the same threshold-relative
HotSpot meaning. Raw SST is insufficient. Region IDs are read as literal text, so `001` stays `001`.
The optional path validates the CSV, then forecasts each region from its last 365 observed days; it does not
pretend unlabelled future dates provide evaluation evidence. Leave the path empty to practise on the generated
`byod_example.csv`. Each distinct input file gets its own folder with forecasts (raw, constrained, quantiles,
DHW components) and a receipt recording the input digest, units, semantics and model revision.
A historical backtest requires separate future outcomes and a newly declared evaluation design.

In [ ]:
USE_BYOD = False # @param {type:"boolean"}
BYOD_CSV_PATH = "" # @param {type:"string"}
HOTSPOT_SEMANTICS_CONFIRMED = False # @param {type:"boolean"}
if USE_BYOD:
    if not HOTSPOT_SEMANTICS_CONFIRMED:
        raise ValueError("Confirm NOAA-compatible threshold-relative HotSpot semantics in °C.")
    source = pathlib.Path(BYOD_CSV_PATH) if BYOD_CSV_PATH else RUN_ROOT / "byod_example.csv"
    print("BYOD input:", source)
    run_stage("byod", "--csv", str(source.resolve()))
else:
    print("BYOD disabled. The complete default experiment is unchanged.")

## Troubleshooting, glossary and sources

- **GPU unavailable:** select a Colab T4 before Run all. CPU engineering checks do not qualify model execution.
- **Download/integrity failure:** retain the error and retry the same pinned inputs in a fresh runtime; never bypass a digest.
- **Dependency/runtime mismatch:** setup builds an isolated uv-managed Python 3.12.13 whatever the kernel version.
  If that download fails, retry in a fresh runtime and preserve the log; do not edit the pinned requirements.
- **Out of memory or excessive runtime:** preserve the failure; do not quietly omit regions, change precision or replace predictions.
- **Missing context:** inspect origin_manifest.csv; missing data is not zero stress.
- **BYOD refusal:** check date uniqueness/continuity, finite nonnegative values, units and target semantics.

**MAE:** mean absolute prediction error. **RMSE:** error emphasizing large misses. **Pinball loss:** quantile error.
**Coverage:** observed fraction inside the stated interval. **Persistence:** repeat the last value. **Macro average:**
equal weight per region, rather than letting more numerous records dominate. **Held out:** not used to tune the experiment.

Credit: NOAA Coral Reef Watch, v3.1 Regional Virtual Station time series, Philippine regions, 1985-01-01–2026-09-26;
accessed 2026-09-28. [Data](https://coralreefwatch.noaa.gov/product/vs/data.php),
[methodology](https://coralreefwatch.noaa.gov/product/5km/methodology.php),
[citation guidance](https://coralreefwatch.noaa.gov/satellite/docs/recommendations_crw_citation.php).
Model: [Amazon Chronos-2](https://huggingface.co/amazon/chronos-2), immutable revision in the export.
Explore related DIMER model pipelines after this activity; the same validation and comparison habits apply.